# 15 — Temporal-window and composite-score sensitivity

Evaluate how temporal averaging and score formulation affect the interpretation of the native-daily results. Run after Notebooks 13 and 14, which export the Samar–Leyte sensitivity matrix and five-sub-grid daily results.

## Key takeaways

Separate direct-date support from window-supported estimates. Inspect the fixed-configuration window comparison before the event panels and score-form sensitivity. Additional historical workflows are optional.

## Run order and exports

Run the cells in order from the repository or notebooks/ folder. Figures use outputs/figures/main/, supplementary/ or diagnostics/. Tables use the matching outputs/tables/ folders. Every static export writes HTML, SVG, PNG and JPEG from the same layout. HTML includes Plotly for offline viewing. JPEG preserves the PNG's pixel dimensions and composites transparency on white.

## 1. Sensitivity inputs

Load the same regional data registry and read the saved Samar–Leyte window sweep and five-sub-grid daily sweep. Run Notebooks 13 and 14 first. Both sweeps are read from `outputs/tables/supplementary/`.

In [ ]:
from io import BytesIO
from PIL import Image as PILImage
import json
import plotly.io as pio

from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

working_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (
        directory
        for directory in (working_dir, *working_dir.parents)
        if (directory / "notebooks").is_dir() and (directory / "datasets").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Open the kernel from the repository or its notebooks folder. "
        "The repository must contain sibling notebooks/ and datasets/ folders."
    )


ROOT = PROJECT_ROOT
OUTPUT_ROOT = PROJECT_ROOT / "outputs"
FIG_MAIN = OUTPUT_ROOT / "figures" / "main"
FIG_SUPP = OUTPUT_ROOT / "figures" / "supplementary"
FIG_DIAGNOSTIC = OUTPUT_ROOT / "figures" / "diagnostics"
TABLE_MAIN = OUTPUT_ROOT / "tables" / "main"
TABLE_SUPP = OUTPUT_ROOT / "tables" / "supplementary"
TABLE_DIAGNOSTIC = OUTPUT_ROOT / "tables" / "diagnostics"
for folder in (FIG_MAIN, FIG_SUPP, FIG_DIAGNOSTIC, TABLE_MAIN, TABLE_SUPP, TABLE_DIAGNOSTIC):
    folder.mkdir(parents=True, exist_ok=True)

# Match Notebook 04's multi-subgrid analysis period and input locations.
DATE_START = pd.Timestamp("2012-01-19")
DATE_END = pd.Timestamp("2024-12-31")
VALIDATION_DATES = pd.date_range(DATE_START, DATE_END, freq="D", name="date")
NTL_FOLDER = ROOT / "datasets" / "vnp46a2" / "visayas_ghsl_masked_daily_stats"
NGCP_XLSX_PATH = ROOT / "datasets" / "ngcp" / "NGCP_hourly_load.xlsx"

WINDOWS = np.arange(1, 31)
WINDOW_CHECKPOINTS = [1, 4, 7, 14, 21, 30]
SC_GRID = np.arange(0, 101, 1, dtype=float)  # exact search and CSV output
MAIN_TAUS = np.arange(0, 101, 5, dtype=float)  # retain the paper's display interval
SUPP_TAUS = MAIN_TAUS.copy()                   # consistent figures throughout
PRIMARY_REGION = "Samar-Leyte"
EVENTS = [
    ("Haiyan", pd.Timestamp("2013-11-08"), "2013-05-01", "2014-05-31"),
    ("Rai", pd.Timestamp("2021-12-16"), "2021-07-01", "2022-07-31"),
]

GROUPS = [
    ("G8", "10-30", "All Land", "#4E79A7"),
    ("G7", "11-30", "Rural Settlements", "#A8CE74"),
    ("G6", "12-30", "Low–Mid Rural", "#7BAF54"),
    ("G5", "13-30", "Built-Up / Rural Mix", "#375623"),
    ("G4", "21-30", "Urban Fringe", "#D9B900"),
    ("G3", "22-30", "Core & Inner", "#E6A23C"),
    ("G2", "23-30", "Dense Urban", "#A87000"),
    ("G1", "30", "Urban Core", "#D65A4A"),
]
GROUP_LOOKUP = {
    code: {"suffix": suffix, "label": label, "color": color}
    for code, suffix, label, color in GROUPS
}
ORDERED_MASKS = [f"G{i}" for i in range(1, 9)]
MASK_COLORS = {code: GROUP_LOOKUP[code]["color"] for code in ORDERED_MASKS}

REGION_RENAME = {
    "CebuProvince": "Cebu",
    "SamarLeyte": "Samar-Leyte",
    "NegrosIsland": "Negros",
    "PanayIsland": "Panay",
    "BoholProvince": "Bohol",
}
REGION_ORDER = ["Samar-Leyte", "Bohol", "Cebu", "Negros", "Panay"]
GROUP_RENAME = {"30-only": "30"}
SHEET_MAP = {
    "Panay": "PANAY HOURLY LOAD 2013-2024",
    "Negros": "NEGROS HOURLY LOAD 2013-2024",
    "Samar-Leyte": "LEY-SAM HOURLY LOAD 2013-2024",
    "Cebu": "CEBU HOURLY LOAD 2013-2024",
    "Bohol": "BOHOL LOAD 2013-2024",
}

PLOT_TEMPLATE = "plotly_white"
INK = "#233B60"
GRID = "#E8EDF3"
R_CONTRAST_SCALE = "jet_r"
N_CONTRAST_SCALE = "Viridis"
SCORE_CONTRAST_SCALE = "Turbo"
DIVERGING_SCALE = [[0.00, "#B2182B"], [0.50, "#F7F7F7"], [1.00, "#2166AC"]]
POSITIVE_SCALE = "Viridis"
R_COLOUR_POWER = 2.0
N_COLOUR_POWER = 3.0
R_TICKS = np.array([-1.0, -0.5, 0.0, 0.5, 0.75, 1.0])
R_COLOUR_TICKS = np.sign(R_TICKS) * np.abs(R_TICKS) ** R_COLOUR_POWER
N_TICKS = np.array([0, 50, 75, 90, 100], dtype=float)
N_COLOUR_TICKS = (N_TICKS / 100) ** N_COLOUR_POWER

def export_figure(
    fig, output_path, width=None, height=None, scale=3,
    export_html=True, force_date_axis=False, tickformat="%Y", static=True,
):
    """Export one figure layout as HTML, SVG, PNG and JPEG."""
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    figure = go.Figure(json.loads(pio.to_json(fig, validate=True)))
    figure.update_layout(
        width=width if width is not None else figure.layout.width or 1200,
        height=height if height is not None else figure.layout.height or 600,
    )
    if force_date_axis:
        figure.update_xaxes(type="date", tickformat=tickformat)

    if static:
        png_bytes = figure.to_image(format="png", scale=scale)
        svg_bytes = figure.to_image(format="svg", scale=1)
        with PILImage.open(BytesIO(png_bytes)) as image:
            rgba = image.convert("RGBA")
            jpeg = PILImage.new("RGB", rgba.size, "white")
            jpeg.paste(rgba, mask=rgba.getchannel("A"))
        buffer = BytesIO()
        jpeg.save(buffer, format="JPEG", quality=95, subsampling=0)
        output_path.with_suffix(".png").write_bytes(png_bytes)
        output_path.with_suffix(".svg").write_bytes(svg_bytes)
        output_path.with_suffix(".jpeg").write_bytes(buffer.getvalue())
    if export_html:
        figure.write_html(
            str(output_path.with_suffix(".html")),
            include_plotlyjs=True, full_html=True,
        )
    return fig
RUN_ADDITIONAL_DIAGNOSTICS = False

In [ ]:
def parse_filename(filepath):
    name = Path(filepath).name.replace("_stats.csv", "")
    parts = name.split("_DNBBRDF_class")
    if len(parts) != 2:
        return "Unknown", "Unknown"
    region, group = parts
    return REGION_RENAME.get(region, region), GROUP_RENAME.get(group, group)


def read_multiregion_ntl(folder):
    csv_files = sorted(Path(folder).glob("*.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No NTL CSV files found in {folder.relative_to(PROJECT_ROOT)}")

    frames = []
    for path in csv_files:
        region, group = parse_filename(path)
        if region == "Unknown":
            continue
        try:
            frame = pd.read_csv(path)
        except Exception:
            frame = pd.read_csv(path, dtype=str, engine="python", on_bad_lines="skip")

        if "date" not in frame.columns and "system:index" in frame.columns:
            frame["date"] = (
                frame["system:index"].astype(str)
                .str.extract(r"(\d{4}_\d{2}_\d{2})")[0]
                .str.replace("_", "-", regex=False)
            )
        if "date" not in frame.columns:
            raise ValueError(f"{path.name}: neither date nor system:index is available")

        frame["Region"] = region
        frame["Group"] = group
        frame["source_file"] = path.relative_to(PROJECT_ROOT).as_posix()
        frames.append(frame)

    if not frames:
        raise RuntimeError(f"No recognised multi-region NTL files were loaded from {folder.relative_to(PROJECT_ROOT)}")

    panel = pd.concat(frames, ignore_index=True)
    panel["date"] = pd.to_datetime(panel["date"], errors="coerce").dt.normalize()
    panel = panel[panel["date"].between(DATE_START, DATE_END)].copy()
    panel["Valid_px"] = pd.to_numeric(panel["Valid_px"], errors="coerce")
    panel["Mean_by_sum"] = pd.to_numeric(panel["Mean_by_sum"], errors="coerce")
    return panel.sort_values(["Region", "Group", "date"]).reset_index(drop=True)


def mask_frame(group_frame):
    data = group_frame.copy().drop_duplicates("date").set_index("date").sort_index()
    valid = data["Valid_px"]
    raw = data["Mean_by_sum"].where(valid > 0)
    denominator = valid.loc[raw.notna()].max()
    if not np.isfinite(denominator) or denominator <= 0:
        raise ValueError("No valid direct observations for this region–mask combination")
    p95 = raw.quantile(0.95)
    return pd.DataFrame({
        "raw": raw,
        "ntl": raw.clip(upper=p95),
        "SC": 100 * valid / denominator,
        "Valid_px": valid,
    }).reindex(VALIDATION_DATES)


def load_ngcp_grid_1am_flat(xlsx_path, sheet_map):
    series_by_region = {}
    selected_columns = {}
    for region, sheet_name in sheet_map.items():
        frame = pd.read_excel(xlsx_path, sheet_name=sheet_name, header=1)
        frame.columns = [str(column).strip() for column in frame.columns]
        date_col = frame.columns[0]
        frame = frame.rename(columns={date_col: "DATE"})
        frame["DATE"] = pd.to_datetime(frame["DATE"], errors="coerce").dt.normalize()
        frame = frame.dropna(subset=["DATE"]).set_index("DATE")
        frame.columns = [str(column).strip() for column in frame.columns]

        possible = ["1", "01", "1.0", "Hour 1", "HOUR 1", "1AM", "1am"]
        hour_col = next((column for column in possible if column in frame.columns), None)
        if hour_col is None:
            numeric_like = [
                column for column in frame.columns
                if str(column).replace(".0", "").isdigit()
            ]
            hour_col = numeric_like[0] if numeric_like else None
        if hour_col is None and len(frame.columns):
            hour_col = frame.columns[0]
        if hour_col is None:
            raise KeyError(f"No usable 01:00 load column in sheet {sheet_name}")

        values = pd.to_numeric(frame[hour_col], errors="coerce")
        values = values.groupby(level=0).mean().reindex(VALIDATION_DATES)
        series_by_region[region] = values.rename(region)
        selected_columns[region] = str(hour_col)

    return pd.DataFrame(series_by_region), selected_columns


NTL_PANEL = read_multiregion_ntl(NTL_FOLDER)
NGCP_PANEL, NGCP_COLUMNS = load_ngcp_grid_1am_flat(NGCP_XLSX_PATH, SHEET_MAP)

DATA_REGISTRY = {}
registry_rows = []
for region in REGION_ORDER:
    masks = {}
    source_files = []
    for code in ORDERED_MASKS:
        group = GROUP_LOOKUP[code]["suffix"]
        subset = NTL_PANEL[
            NTL_PANEL["Region"].eq(region) & NTL_PANEL["Group"].eq(group)
        ].copy()
        if subset.empty:
            raise FileNotFoundError(
                f"{region}: GHSL group {group} ({code}) is absent from {NTL_FOLDER.relative_to(PROJECT_ROOT)}"
            )
        masks[code] = mask_frame(subset)
        source_files.extend(subset["source_file"].dropna().unique().tolist())

    load = NGCP_PANEL[region].copy()
    DATA_REGISTRY[region] = {"masks": masks, "load": load}
    registry_rows.append({
        "region": region,
        "ntl_files": len(set(source_files)),
        "ghsl_masks": len(masks),
        "ngcp_sheet": SHEET_MAP[region],
        "load_column": NGCP_COLUMNS[region],
        "load_dates": int(load.notna().sum()),
    })

REGISTRY_TABLE = pd.DataFrame(registry_rows)
display(REGISTRY_TABLE)
REGISTRY_TABLE.to_csv(TABLE_SUPP / "data_registry.csv", index=False)

In [ ]:
SENS_PRIMARY = pd.read_csv(TABLE_SUPP / "samar_leyte_mask_sc_window_sensitivity.csv.gz")
DAILY_ALL = pd.read_csv(TABLE_SUPP / "all_subgrids_daily_mask_sc_results.csv.gz")

## 2. Temporal support and alignment as the window widens

The fixed-configuration figure compares correlations on the same directly paired dates and separately displays coverage from window-supported estimates. The additional profiles and checkpoint surfaces show where the result changes across masks and SC thresholds.

In [ ]:
# ============================================================
# Figure 10 — Window sensitivity at fixed W=1 configurations
# Run after the Notebook 10 cells that build DATA_REGISTRY and DAILY_ALL.
# ============================================================

CHECKPOINTS = [1, 4, 7, 14, 21, 30]
REGIONS = ["Samar-Leyte", "Bohol", "Cebu", "Negros", "Panay"]

# Use the same threshold grid and ranking rule as Figures 6 and 8.
selected = (
    DAILY_ALL.loc[
        DAILY_ALL["window_days"].eq(1)
        & DAILY_ALL["tau"].isin(range(5, 100, 5))
        & DAILY_ALL["region"].isin(REGIONS)
    ]
    .dropna(subset=["r", "score"])
    .sort_values(
        ["score", "N_pct", "tau", "mask"],
        ascending=[False, False, True, True],
    )
    .drop_duplicates("region")
    .set_index("region")
    .reindex(REGIONS)
)

if selected["mask"].isna().any():
    missing = selected.index[selected["mask"].isna()].tolist()
    raise ValueError(f"No eligible W=1 configuration for {', '.join(missing)}.")

records = []
headings = []

for region in REGIONS:
    mask = selected.at[region, "mask"]
    tau = float(selected.at[region, "tau"])
    headings.append(f"{region}<br><sup>{mask} · τ = {tau:.0f}%</sup>")

    region_data = DATA_REGISTRY[region]
    load = region_data["load"]
    ntl_data = region_data["masks"][mask].reindex(load.index)

    # Reuse the winsorised NTL and SC rule from the native-daily sweep.
    ntl = ntl_data["ntl"].where(ntl_data["SC"].ge(tau))
    load_dates = load.notna()
    direct_pairs = load_dates & ntl.notna()
    n_load = int(load_dates.sum())
    n_direct = int(direct_pairs.sum())

    if n_direct < 3:
        raise ValueError(f"{region}: fewer than three directly paired W=1 dates.")

    direct_pct = 100 * n_direct / n_load
    daily_r = ntl.loc[direct_pairs].corr(load.loc[direct_pairs])

    if (
        n_direct != int(selected.at[region, "n_pairs"])
        or not np.isclose(daily_r, selected.at[region, "r"])
    ):
        raise ValueError(f"{region}: the W=1 inputs do not reproduce the daily sweep.")

    for w in CHECKPOINTS:
        # The registry uses a complete daily calendar.
        ntl_w = ntl.rolling(w, center=True, min_periods=1).mean()
        load_w = load.rolling(w, center=True, min_periods=1).mean()

        # Evaluate Pearson r on the same directly paired dates at every W.
        r_fixed = ntl_w.loc[direct_pairs].corr(
            load_w.loc[direct_pairs], min_periods=3
        )

        # Use the same NGCP observation dates as the coverage denominator.
        estimate_pairs = load_dates & ntl_w.notna() & load_w.notna()
        n_estimates = int(estimate_pairs.sum())

        records.append({
            "region": region,
            "mask": mask,
            "tau": tau,
            "window_days": w,
            "r_fixed_dates": r_fixed,
            "paired_estimate_pct": 100 * n_estimates / n_load,
            "direct_pct": direct_pct,
            "n_direct_pairs": n_direct,
            "n_paired_estimates": n_estimates,
            "n_load_dates": n_load,
        })

WINDOW_CHECKPOINT_TABLE = pd.DataFrame(records)

fig = make_subplots(
    rows=2,
    cols=5,
    shared_xaxes="all",
    shared_yaxes="rows",
    column_titles=headings,
    horizontal_spacing=0.028,
    vertical_spacing=0.10,
)

for heading in fig.layout.annotations:
    heading.font = dict(family="Arial", size=18, color=INK)

blue = "#2171B5"
green = "#238B45"
dark_green = "#00441B"

for col, region in enumerate(REGIONS, start=1):
    profile = WINDOW_CHECKPOINT_TABLE.loc[
        WINDOW_CHECKPOINT_TABLE["region"].eq(region)
    ]

    fig.add_trace(
        go.Scatter(
            x=profile["window_days"],
            y=profile["r_fixed_dates"],
            mode="lines+markers",
            name="Pearson r",
            showlegend=(col == 1),
            line=dict(color=blue, width=2.5),
            marker=dict(
                symbol="circle-open", color=blue, size=8, line=dict(width=1.5)
            ),
            hovertemplate=(
                "W=%{x} days<br>Pearson r on W=1 paired dates=%{y:.3f}"
                "<extra></extra>"
            ),
        ),
        row=1,
        col=col,
    )

    fig.add_trace(
        go.Scatter(
            x=profile["window_days"],
            y=profile["paired_estimate_pct"],
            mode="lines+markers",
            name="Paired-estimate availability",
            showlegend=(col == 1),
            line=dict(color=green, width=2.5),
            marker=dict(
                symbol="circle-open", color=green, size=8, line=dict(width=1.5)
            ),
            hovertemplate=(
                "W=%{x} days<br>Paired-estimate availability=%{y:.1f}%"
                "<extra></extra>"
            ),
        ),
        row=2,
        col=col,
    )

    fig.add_trace(
        go.Scatter(
            x=profile["window_days"],
            y=profile["direct_pct"],
            mode="lines",
            name="Native-daily coverage",
            showlegend=(col == 1),
            line=dict(color=INK, width=1.8, dash="dash"),
            hovertemplate="Native-daily coverage=%{y:.1f}%<extra></extra>",
        ),
        row=2,
        col=col,
    )

    direct_pct = float(profile["direct_pct"].iloc[0])
    fig.add_annotation(
        x=CHECKPOINTS[-1],
        y=direct_pct,
        text=f"N<sub>1</sub> = {direct_pct:.1f}%",
        xanchor="right",
        yanchor="bottom",
        yshift=4,
        showarrow=False,
        font=dict(family="Arial", size=13, color=INK),
        row=2,
        col=col,
    )

fig.update_xaxes(
        tickmode="array",
        tickvals=CHECKPOINTS,
        showgrid=True,
        gridcolor="rgba(0, 0, 0, 0.09)",
        gridwidth=1,
)
fig.update_xaxes(showticklabels=True, ticks="outside", ticklen=4, row=2)

fig.update_yaxes(
    range=[0.25, 1.00],
    tickvals=[0.25, 0.50, 0.75, 1.00],
    tickformat=".2f",
    gridcolor=GRID,
    zeroline=False,
    row=1,
)
fig.update_yaxes(
    range=[0, 110],
    tickvals=[0, 25, 50, 75, 100],
    gridcolor=GRID,
    zeroline=False,
    row=2,
)
fig.update_yaxes(title_text="Pearson r", row=1, col=1)
fig.update_yaxes(title_text="N (%)", row=2, col=1)

fig.add_annotation(
    x=0.5,
    y=-0.13,
    xref="paper",
    yref="paper",
    text="Centred calendar window, W (days)",
    showarrow=False,
    font=dict(size=17),
)

width, height = 1500, 630
fig.update_layout(
    template=PLOT_TEMPLATE,
    width=width,
    height=height,
    margin=dict(l=30, r=30, t=80, b=30),
    legend=dict(orientation="h", x=0.8, xanchor="center", y=0.55),
    font=dict(family="Arial", color=INK, size=15),
    paper_bgcolor="white",
    plot_bgcolor="white",
)

fig.update_layout(showlegend=False)

output_dir = FIG_MAIN
output_dir.mkdir(parents=True, exist_ok=True)
stem = "fig_10_fixed_configuration_window_sensitivity"
WINDOW_CHECKPOINT_TABLE.to_csv(TABLE_SUPP / f"{stem}.csv", index=False)

if WINDOW_CHECKPOINT_TABLE["r_fixed_dates"].lt(0.25).any():
    print("Pearson r values below 0.25 are retained in the CSV.")

fig.show()
MAIN_WINDOW_EFFECT = export_figure(fig, (output_dir) / (stem), width=width, height=height)

In [ ]:
DAILY_PREVIEW = SENS_PRIMARY[SENS_PRIMARY.window_days.eq(1)].sort_values(
    ["score", "N_pct", "tau", "mask"], ascending=[False, False, True, True]
)
REFERENCE = DAILY_PREVIEW.iloc[0]
REFERENCE_MASK = REFERENCE["mask"]
REFERENCE_TAU = float(REFERENCE["tau"])
WINDOW_PROFILE = SENS_PRIMARY[
    SENS_PRIMARY["mask"].eq(REFERENCE_MASK)
    & SENS_PRIMARY["tau"].eq(REFERENCE_TAU)
].sort_values("window_days")
WINDOW_CHECKPOINT_TABLE = WINDOW_PROFILE[
    WINDOW_PROFILE.window_days.isin(WINDOW_CHECKPOINTS)
][[
    "window_days", "mask", "tau", "r", "n_pairs", "N_pct",
    "direct_pct", "mean_direct_days", "mean_window_support_pct",
]]
display(WINDOW_CHECKPOINT_TABLE.round(3))
WINDOW_CHECKPOINT_TABLE.to_csv(TABLE_MAIN / "section1_window_checkpoint_table.csv", index=False)

fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True, vertical_spacing=.12,
    subplot_titles=["Pearson r", "Paired-estimate N% and direct-observation N%"],
)
fig.add_trace(go.Scatter(
    x=WINDOW_PROFILE.window_days, y=WINDOW_PROFILE.r, mode="lines+markers",
    name="Pearson r", line=dict(color="#2952CC", width=2.8),
    hovertemplate="W=%{x} d<br>r=%{y:.3f}<extra></extra>",
), row=1, col=1)
fig.add_trace(go.Scatter(
    x=WINDOW_PROFILE.window_days, y=WINDOW_PROFILE.N_pct, mode="lines+markers",
    name="Paired-estimate N%", line=dict(color="#D65A4A", width=2.8),
), row=2, col=1)
fig.add_trace(go.Scatter(
    x=WINDOW_PROFILE.window_days, y=WINDOW_PROFILE.direct_pct, mode="lines",
    name="Direct-observation N%", line=dict(color=INK, width=2.2, dash="dash"),
), row=2, col=1)
for row, metric, color in [(1, "r", "#2952CC"), (2, "N_pct", "#D65A4A")]:
    check = WINDOW_PROFILE[WINDOW_PROFILE.window_days.isin(WINDOW_CHECKPOINTS)]
    fig.add_trace(go.Scatter(
        x=check.window_days, y=check[metric], mode="markers+text",
        text=[f"W={int(w)}" if w in (1, 4, 30) else "" for w in check.window_days],
        textposition="top center", showlegend=False,
        marker=dict(symbol="circle-open", size=10, color=color, line=dict(width=2)),
        hoverinfo="skip",
    ), row=row, col=1)
fig.update_xaxes(title_text="Centered common calendar window (days)", tickvals=WINDOW_CHECKPOINTS, row=2, col=1)
fig.update_yaxes(title_text="Pearson r", range=[-1, 1.03], gridcolor=GRID, row=1, col=1)
fig.update_yaxes(title_text="N%", range=[0, 102], gridcolor=GRID, row=2, col=1)
fig.update_layout(
    template=PLOT_TEMPLATE, width=1250, height=760,
    title=f"Window effect after fixing {REFERENCE_MASK} · SC τ={REFERENCE_TAU:.0f}% from W=1",
    legend=dict(orientation="h", y=-.12), margin=dict(l=80, r=40, t=90, b=95),
    font=dict(family="Arial", color=INK, size=17),
    paper_bgcolor="white", plot_bgcolor="white",
)
fig.show()
MAIN_WINDOW_EFFECT = export_figure(fig, (FIG_MAIN) / ("section1_window_effect_summary"), width=1250, height=760)

In [ ]:
# ============================================================
# Supplementary checkpoint heatmaps
# W = 1, 4, 7, 14, 21, and 30 days
# ============================================================

DISPLAY_TAUS_5 = np.arange(5, 100, 5)
CHECKPOINT_WINDOWS = [1, 4, 7, 14, 21, 30]

FULL_MASK_LABELS = {
    code: f"{code}: {GROUP_LOOKUP[code]['label']}"
    for code in ORDERED_MASKS
}


def checkpoint_heatmaps(metric):
    is_r = metric == "r"
    metric_title = "Pearson r" if is_r else "Paired-estimate N%"
    colorscale = "Jet" if is_r else "Viridis"
    cmin, cmax = (0, 1) if is_r else (0, 100)
    ticks = [0, .25, .5, .75, 1] if is_r else [0, 25, 50, 75, 100]
    ticktext = ["0", ".25", ".50", ".75", "1"] if is_r else ["0", "25", "50", "75", "100"]

    fig = make_subplots(
        rows=1, cols=6, shared_yaxes=True,
        horizontal_spacing=.025,
        subplot_titles=[f"W = {w} d" for w in CHECKPOINT_WINDOWS],
    )

    for heading in fig.layout.annotations:
        heading.font = dict(family="Arial", size=19, color=INK)

    for col, window in enumerate(CHECKPOINT_WINDOWS, start=1):
        subset = SENS_PRIMARY[SENS_PRIMARY.window_days.eq(window)]

        matrix = (
            subset.pivot(index="mask", columns="tau", values=metric)
            .reindex(index=ORDERED_MASKS, columns=DISPLAY_TAUS_5)
        )

        raw = matrix.to_numpy(dtype=float)
        colour_values = np.clip(raw, 0, 1) if is_r else raw
        value_hover = (
            "Pearson r=%{customdata:.3f}"
            if is_r else
            "Paired-estimate N=%{customdata:.1f}%"
        )

        fig.add_trace(
            go.Heatmap(
                x=matrix.columns,
                y=[FULL_MASK_LABELS[code] for code in matrix.index],
                z=colour_values,
                customdata=raw,
                coloraxis="coloraxis",
                xgap=.8,
                ygap=.8,
                hoverongaps=False,
                hovertemplate=(
                    f"W={window} days<br>"
                    "Mask=%{y}<br>"
                    "SC threshold τ=%{x:.0f}%<br>"
                    + value_hover
                    + "<extra></extra>"
                ),
            ),
            row=1, col=col,
        )

        fig.update_xaxes(
            range=[2.5, 97.5],
            title_text="SC threshold, τ (%)",
            title_font=dict(family="Arial", size=17, color=INK),
            tickvals=[20, 40, 60, 80],
            ticktext=["20", "40", "60", "80"],
            tickfont=dict(family="Arial", size=15, color=INK),
            showline=True, mirror=True,
            linecolor=INK, linewidth=1.4,
            ticks="outside", ticklen=5,
            row=1, col=col,
        )

        fig.update_yaxes(
            title_text="GHSL thematic mask" if col == 1 else None,
            title_font=dict(family="Arial", size=18, color=INK),
            tickfont=dict(family="Arial", size=14, color=INK),
            showticklabels=(col == 1),
            showline=True, mirror=True,
            linecolor=INK, linewidth=1.4,
            ticks="outside", ticklen=5,
            row=1, col=col,
        )

    # Centred metric heading.
    fig.add_annotation(
        x=.5, y=1.13,
        xref="paper", yref="paper",
        text=metric_title,
        showarrow=False,
        xanchor="center", yanchor="middle",
        font=dict(family="Arial", size=23, color=INK),
    )

    # Compact colour bar at the upper-right.
    colorbar = dict(
        orientation="h",
        x=.91, xanchor="center",
        y=1.13, yanchor="middle",
        len=.13, thickness=25,
        tickvals=ticks, ticktext=ticktext,
        tickfont=dict(family="Arial", size=14, color=INK),
        ticks="outside", ticklen=4,
        outlinecolor=INK, outlinewidth=1,
    )

    fig.update_layout(
        template="plotly_white",
        width=2300, height=570,
        title=None,
        coloraxis=dict(
            colorscale=colorscale,
            cmin=cmin, cmax=cmax,
            colorbar=colorbar,
        ),
        margin=dict(l=225, r=60, t=105, b=85),
        font=dict(family="Arial", color=INK, size=16),
        paper_bgcolor="white",
        plot_bgcolor="#E9EDF2",
    )

    fig.show()

    tag = "r" if is_r else "N_pct"
    return export_figure(fig, (FIG_SUPP) / (f"section1_checkpoint_heatmaps_{tag}"), width=2300, height=570)


SUPP_CHECKPOINT_R = checkpoint_heatmaps("r")
SUPP_CHECKPOINT_N = checkpoint_heatmaps("N_pct")

In [ ]:
# ============================================================
# Supplementary sensitivity surface — compact final version
# ============================================================

TAUS_5 = np.arange(5, 100, 5)
WINDOWS_1_30 = np.arange(1, 31)

GAP = 0.17
LEFT_CENTER = (1 - GAP) / 4
RIGHT_CENTER = 1 - LEFT_CENTER

FONT = dict(family="Arial", color=INK)
AXIS_LINE = dict(showline=True, mirror=True, linecolor=INK, linewidth=1.4,
                 ticks="outside", ticklen=5, tickwidth=1.2)


def sensitivity_matrix(code, metric):
    return (
        SENS_PRIMARY[SENS_PRIMARY["mask"].eq(code)]
        .pivot(index="window_days", columns="tau", values=metric)
        .reindex(index=WINDOWS_1_30, columns=TAUS_5)
    )


def mask_progression_surface():
    fig = make_subplots(
        rows=8, cols=2, shared_xaxes=True, shared_yaxes=True,
        horizontal_spacing=GAP, vertical_spacing=0.022,
    )

    metric_specs = [
        ("r", 1, "coloraxis", "Pearson r"),
        ("N_pct", 2, "coloraxis2", "Paired-estimate N%"),
    ]

    for row, code in enumerate(ORDERED_MASKS, start=1):
        for metric, col, coloraxis, label in metric_specs:
            matrix = sensitivity_matrix(code, metric)
            raw = matrix.to_numpy(dtype=float)

            # Negative r values receive the r=0 colour only.
            colour_values = np.clip(raw, 0, 1) if metric == "r" else raw

            fig.add_trace(
                go.Heatmap(
                    x=matrix.columns, y=matrix.index,
                    z=colour_values, customdata=raw,
                    coloraxis=coloraxis, xgap=0.7, ygap=0,
                    hoverongaps=False,
                    hovertemplate=(
                        f"{code} · {GROUP_LOOKUP[code]['label']}<br>"
                        "SC τ=%{x:.0f}%<br>W=%{y:.0f} days<br>"
                        f"{label}=%{{customdata:.3f}}<extra></extra>"
                    ),
                ),
                row=row, col=col,
            )

        for col in (1, 2):
            fig.update_xaxes(
                range=[2.5, 97.5],
                tickvals=[20, 40, 60, 80],
                ticktext=["20", "40", "60", "80"],
                showticklabels=(row == 8),
                title_text="Spatial completeness threshold, τ (%)" if row == 8 else None,
                title_font={**FONT, "size": 21},
                tickfont={**FONT, "size": 16},
                tickangle=0, row=row, col=col, **AXIS_LINE,
            )

            fig.update_yaxes(
                range=[0.5, 30.5],
                tickvals=[1, 4, 7, 14, 21, 30],
                showticklabels=(col == 1),
                title_text="W (days)" if col == 1 else None,
                title_font={**FONT, "size": 17},
                tickfont={**FONT, "size": 15},
                row=row, col=col, **AXIS_LINE,
            )

        # Horizontal GHSL label centred in the gap.
        fig.add_annotation(
            x=0.5, y=1 - (row - 0.5) / 8,
            xref="paper", yref="paper",
            text=f"{code} · {GROUP_LOOKUP[code]['label']}",
            showarrow=False, xanchor="center", yanchor="middle",
            font={**FONT, "size": 17},
        )

    # Column headings centred over their respective panels.
    for x, text in [
        (LEFT_CENTER, "Pearson r"),
        (RIGHT_CENTER, "Paired-estimate N%"),
    ]:
        fig.add_annotation(
            x=x, y=1.075, xref="paper", yref="paper",
            text=text, showarrow=False,
            xanchor="center", yanchor="middle",
            font={**FONT, "size": 25},
        )

    shared_colorbar = dict(
        orientation="h", y=1.025, yanchor="middle",
        len=0.16, thickness=28,
        tickfont={**FONT, "size": 15},
        ticks="outside", ticklen=4,
        outlinecolor=INK, outlinewidth=1,
    )

    fig.update_layout(
        coloraxis=dict(
            colorscale="Jet", cmin=0, cmax=1,
            colorbar={
                **shared_colorbar,
                "x": LEFT_CENTER, "xanchor": "center",
                "tickvals": [0, .25, .5, .75, 1],
                "ticktext": ["0", ".25", ".50", ".75", "1"],
            },
        ),
        coloraxis2=dict(
            colorscale="Viridis", cmin=0, cmax=100,
            colorbar={
                **shared_colorbar,
                "x": RIGHT_CENTER, "xanchor": "center",
                "tickvals": [0, 25, 50, 75, 100],
                "ticktext": ["0", "25", "50", "75", "100"],
            },
        ),
        template="plotly_white",
        width=2300, height=1500,
        title=None,
        margin=dict(l=150, r=95, t=155, b=125),
        font={**FONT, "size": 17},
        paper_bgcolor="white",
        plot_bgcolor="#E9EDF2",
    )

    fig.show()

    return export_figure(fig, (FIG_SUPP) / ("section1_mask_progression_r_and_paired_N_5pct"), width=2300, height=1500)


SUPP_MASK_PROGRESSION = mask_progression_surface()

In [ ]:
SLICE_TAUS = [0, 10, 25, 50, 75, 90]
LABEL_WINDOWS = [4, 14, 30]


def plot_window_slices(metric, ylabel, folder, stem):
    is_r = metric == "r"
    is_n = metric == "N_pct"
    fig = make_subplots(
        rows=2, cols=3, shared_xaxes=True, shared_yaxes=True,
        horizontal_spacing=.06, vertical_spacing=.16,
        subplot_titles=[f"SC τ={tau}%" for tau in SLICE_TAUS],
    )
    for heading in fig.layout.annotations:
        heading.font = dict(family="Arial", size=19, color=INK)

    for panel, tau in enumerate(SLICE_TAUS):
        row, col = panel // 3 + 1, panel % 3 + 1
        for code in ORDERED_MASKS:
            q = SENS_PRIMARY[
                SENS_PRIMARY["mask"].eq(code) & SENS_PRIMARY["tau"].eq(float(tau))
            ].sort_values("window_days")
            shown = q[metric].where(q[metric] > 0) if is_n else q[metric]
            fig.add_trace(
                go.Scatter(
                    x=q.window_days, y=shown, mode="lines",
                    name=f"{code} {GROUP_LOOKUP[code]['label']}",
                    legendgroup=code, showlegend=(panel == 0), connectgaps=False,
                    line=dict(color=MASK_COLORS[code], width=2.4),
                    customdata=np.column_stack([
                        q.N_pct, q.direct_pct, q.mean_window_support_pct,
                    ]),
                    hovertemplate=(
                        f"W=%{{x}} d<br>{ylabel}=%{{y:.3f}}"
                        "<br>Paired estimates=%{customdata[0]:.1f}%"
                        "<br>Direct dates=%{customdata[1]:.1f}%"
                        "<br>Mean direct support=%{customdata[2]:.1f}%"
                        "<extra>%{fullData.name}</extra>"
                    ),
                ), row=row, col=col,
            )

        for code in ("G1", "G8"):
            q = SENS_PRIMARY[
                SENS_PRIMARY["mask"].eq(code)
                & SENS_PRIMARY["tau"].eq(float(tau))
                & SENS_PRIMARY.window_days.isin(LABEL_WINDOWS)
            ].sort_values("window_days")
            for _, point in q.iterrows():
                value = point[metric]
                if not np.isfinite(value) or (is_n and value <= 0):
                    continue
                above = code == ("G8" if is_n else "G1")
                fig.add_annotation(
                    x=point.window_days, y=value,
                    text=f"{value:.1f}%" if is_n else f"{value:.2f}",
                    showarrow=True, arrowhead=0, arrowwidth=.75,
                    arrowcolor="#A9B9C9", ax=0, ay=-22 if above else 22,
                    font=dict(size=13, color=MASK_COLORS[code]),
                    bgcolor="white", bordercolor=MASK_COLORS[code],
                    borderwidth=.8, borderpad=2, row=row, col=col,
                )

        fig.update_xaxes(
            range=[1, 30], tickvals=[1, 4, 7, 14, 21, 30],
            gridcolor=GRID, showline=True, linecolor=INK, linewidth=1.4,
            ticks="outside", tickfont=dict(size=15), row=row, col=col,
        )
        y_style = dict(
            gridcolor=GRID, zeroline=False, showline=True, linecolor=INK,
            linewidth=1.4, ticks="outside", tickfont=dict(size=15),
            showticklabels=(col == 1),
        )
        if is_n:
            y_style.update(
                type="log", range=[-2, 2.15],
                tickvals=[.01, .1, 1, 10, 100],
                ticktext=["0.01", "0.1", "1", "10", "100"],
            )
        else:
            y_style.update(range=[-1.05, 1.05], tickvals=[-1, -.5, 0, .5, 1])
        fig.update_yaxes(row=row, col=col, **y_style)

    fig.add_annotation(
        x=-.065, y=.5, xref="paper", yref="paper",
        text="N% paired estimates (log scale)" if is_n else ylabel,
        textangle=-90, showarrow=False, font=dict(size=18, color=INK),
    )
    fig.add_annotation(
        x=.5, y=-.09, xref="paper", yref="paper",
        text="Centred common calendar window (days)",
        showarrow=False, font=dict(size=18, color=INK),
    )
    fig.update_layout(
        template=PLOT_TEMPLATE, width=1800, height=850,
        legend=dict(x=1.015, y=1, xanchor="left", yanchor="top", font=dict(size=15)),
        margin=dict(l=95, r=230, t=100, b=95),
        font=dict(family="Arial", color=INK, size=17),
        paper_bgcolor="white", plot_bgcolor="white",
    )
    fig.show()
    return export_figure(fig, (folder) / (stem), width=1800, height=850)


MAIN_WINDOW_R_SLICES = plot_window_slices(
    "r", "Pearson r", FIG_MAIN, "section1_window_trajectories_r"
)
SUPP_WINDOW_N_SLICES = plot_window_slices(
    "N_pct", "N% paired estimates", FIG_SUPP, "section1_window_trajectories_N"
)

## 3. Haiyan and Rai signal representation

The event panels preserve the source windows and compare how centred means change the shape and coverage of the NTL and NGCP series. The selected daily configuration is held fixed across the displayed windows.

In [ ]:
# ============================================================
# Event-period NTL–NGCP alignment
# Main: W = 1, 4, 30
# Supplement: W = 1, 4, 7, 14, 30
# ============================================================

TAUS_5 = np.arange(5, 100, 5)

EVENT_CASES = [
    dict(event="Haiyan", region="Samar-Leyte", date="2013-11-08",
         left="2013-05-01", right="2014-05-31"),
    dict(event="Rai", region="Bohol", date="2021-12-16",
         left="2021-07-01", right="2022-07-31"),
]


def best_daily_config(region):
    q = DAILY_ALL[
        DAILY_ALL.region.eq(region)
        & DAILY_ALL.window_days.eq(1)
        & DAILY_ALL.tau.isin(TAUS_5)
    ]
    if q.empty:
        raise ValueError(f"No daily configuration found for {region}")

    return q.sort_values(
        ["score", "N_pct", "r", "tau", "mask"],
        ascending=[False, False, False, True, True],
    ).iloc[0]


BEST_EVENT_CONFIGS = {
    case["region"]: best_daily_config(case["region"])
    for case in EVENT_CASES
}


def paired_zscores(ntl, load):
    paired = pd.DataFrame({"ntl": ntl, "load": load}).dropna()

    if len(paired) < 3 or paired.ntl.std(ddof=0) == 0 or paired.load.std(ddof=0) == 0:
        return (
            pd.Series(np.nan, index=ntl.index),
            pd.Series(np.nan, index=load.index),
        )

    return (
        (ntl - paired.ntl.mean()) / paired.ntl.std(ddof=0),
        (load - paired.load.mean()) / paired.load.std(ddof=0),
    )


def event_signal_figure(windows, folder, stem, width):
    ncols = len(windows)
    annotation_size = 13 if ncols == 5 else 14

    fig = make_subplots(
        rows=2, cols=ncols, shared_yaxes=True,
        horizontal_spacing=.025 if ncols == 5 else .05,
        vertical_spacing=.19,
    )

    for row, case in enumerate(EVENT_CASES, start=1):
        region, event = case["region"], case["event"]
        event_date = pd.Timestamp(case["date"])
        dates = pd.date_range(case["left"], case["right"], freq="D")

        best = BEST_EVENT_CONFIGS[region]
        mask, tau = best["mask"], float(best["tau"])
        mask_data = DATA_REGISTRY[region]["masks"][mask]
        load = DATA_REGISTRY[region]["load"]

        direct_mask = mask_data.ntl.notna() & mask_data.SC.ge(tau)
        qualified_ntl = mask_data.ntl.where(direct_mask)

        for col, w in enumerate(windows, start=1):
            ntl_w = qualified_ntl.rolling(w, min_periods=1, center=True).mean()
            load_w = load.rolling(w, min_periods=1, center=True).mean()
            support_w = qualified_ntl.notna().astype(int).rolling(
                w, min_periods=1, center=True
            ).sum()

            ntl_z, load_z = paired_zscores(ntl_w, load_w)
            event_pairs = pd.DataFrame({
                "ntl": ntl_z.reindex(dates),
                "load": load_z.reindex(dates),
            }).dropna()

            event_r = (
                event_pairs.ntl.corr(event_pairs.load)
                if len(event_pairs) >= 3 else np.nan
            )

            load_available = load.reindex(dates).notna()
            possible_n = int(load_available.sum())
            paired_pct = 100 * len(event_pairs) / possible_n if possible_n else np.nan

            direct_available = (
                direct_mask.reindex(dates).fillna(False).astype(bool)
                & load_available
            )
            direct_pct = (
                100 * int(direct_available.sum()) / possible_n
                if possible_n else np.nan
            )

            support = support_w.reindex(event_pairs.index)
            mean_support = float(support.mean()) if len(support) else np.nan
            r_text = f"{event_r:.2f}" if np.isfinite(event_r) else "—"

            # SC-qualified NTL
            fig.add_trace(
                go.Scatter(
                    x=dates, y=ntl_z.reindex(dates), mode="lines",
                    name="SC-qualified NTL", legendgroup="ntl",
                    showlegend=(row == 1 and col == 1),
                    line=dict(color="#2952CC", width=2.6),
                    connectgaps=False,
                    customdata=support_w.reindex(dates),
                    hovertemplate=(
                        "%{x|%d %b %Y}<br>NTL z = %{y:.2f}"
                        "<br>Direct observations in window = %{customdata:.0f}"
                        "<extra>SC-qualified NTL</extra>"
                    ),
                ),
                row=row, col=col,
            )

            # NGCP load
            fig.add_trace(
                go.Scatter(
                    x=dates, y=load_z.reindex(dates), mode="lines",
                    name="NGCP load at 01:00", legendgroup="load",
                    showlegend=(row == 1 and col == 1),
                    line=dict(color="#202020", width=2.3),
                    connectgaps=False,
                    hovertemplate=(
                        "%{x|%d %b %Y}<br>NGCP z = %{y:.2f}"
                        "<extra>NGCP load at 01:00</extra>"
                    ),
                ),
                row=row, col=col,
            )

            fig.add_vline(
                x=event_date, row=row, col=col,
                line_dash="dash", line_color=INK, line_width=1.5,
            )

            axis_n = (row - 1) * ncols + col
            xref = "x domain" if axis_n == 1 else f"x{axis_n} domain"
            yref = "y domain" if axis_n == 1 else f"y{axis_n} domain"

            # Window is the annotation heading; metrics are secondary.
            fig.add_annotation(
                x=.985, y=.97, xref=xref, yref=yref,
                text=(
                    f"W = {w} days"
                    f"<br><i>r</i><sub>event</sub> = {r_text}"
                    f"<br><i>N</i><sub>paired</sub> = {paired_pct:.1f}%"
                    f" · <i>N</i><sub>direct</sub> = {direct_pct:.1f}%"
                    f"<br>mean support = {mean_support:.1f}/{w} d"
                ),
                showarrow=False, xanchor="right", yanchor="top", align="right",
                bgcolor="rgba(255,255,255,.88)",
                bordercolor="#A9B9C9", borderwidth=.8, borderpad=5,
                font=dict(family="Arial", size=annotation_size, color=INK),
            )

            fig.update_xaxes(
                tickformat="%b\n%Y", nticks=6,
                tickfont=dict(family="Arial", size=13),
                gridcolor=GRID, showline=True, linecolor=INK,
                linewidth=1.2, ticks="outside", ticklen=4,
                row=row, col=col,
            )

        # One centred scientific title per event row.
        fig.add_annotation(
            x=.5, y=1.07 if row == 1 else .50,
            xref="paper", yref="paper",
            text=f"{event} · {region} · {mask} · τ = {tau:.0f}%",
            showarrow=False, xanchor="center", yanchor="middle",
            font=dict(family="Arial", size=21, color=INK),
        )

    fig.update_yaxes(
        range=[-4, 4], tickvals=[-4, -2, 0, 2, 4],
        tickfont=dict(family="Arial", size=14),
        gridcolor=GRID, zeroline=True, zerolinecolor="#A9B9C9",
        showline=True, linecolor=INK, linewidth=1.2,
        ticks="outside", ticklen=4,
    )

    for row in (1, 2):
        fig.update_yaxes(
            title_text="Standardised signal (z)",
            title_font=dict(family="Arial", size=17, color=INK),
            row=row, col=1,
        )

    fig.update_layout(
        template="plotly_white",
        width=width, height=800,
        title=None,

        # Horizontal legend aligned with the first row title.
        legend=dict(
            orientation="h",
            x=.99, xanchor="right",
            y=1.07, yanchor="middle",
            font=dict(family="Arial", size=15, color=INK),
            bgcolor="rgba(255,255,255,.8)",
        ),

        margin=dict(l=100, r=35, t=90, b=65),
        font=dict(family="Arial", color=INK, size=15),
        paper_bgcolor="white", plot_bgcolor="white",
    )

    fig.show()
    return export_figure(fig, (folder) / (stem), width=width, height=800)


# Compact main-manuscript version
MAIN_EVENT_SIGNAL = event_signal_figure(
    windows=[1, 4, 30],
    folder=FIG_MAIN,
    stem="section1_event_alignment_main_W1_W4_W30",
    width=1700,
)

# Complete supplementary sensitivity version
SUPP_EVENT_SIGNAL = event_signal_figure(
    windows=[1, 4, 7, 14, 21, 30],
    folder=FIG_SUPP,
    stem="section1_event_alignment_supp_W1_W4_W7_W14_W30",
    width=2400,
)

## 4. Composite-score formulation

Compare the daily score using positive $r$ and positive $r^2$ on the same regional configuration grid. Inspect changes in the selected mask and threshold alongside the alignment and coverage components.

In [ ]:
# ============================================================
# Supplementary score-form sensitivity
# Compare N×r against N×r²
# ============================================================

TAUS_5 = np.arange(5, 100, 5)

SCORE_SENS = DAILY_ALL[
    DAILY_ALL.window_days.eq(1)
    & DAILY_ALL.tau.isin(TAUS_5)
].copy()

SCORE_SENS["r_pos"] = SCORE_SENS.r.clip(lower=0)
SCORE_SENS["score_Nr"] = (SCORE_SENS.N_pct / 100) * SCORE_SENS.r_pos
SCORE_SENS["score_Nr2"] = (SCORE_SENS.N_pct / 100) * SCORE_SENS.r_pos.pow(2)

for score in ["score_Nr", "score_Nr2"]:
    SCORE_SENS[f"{score}_norm"] = SCORE_SENS.groupby("region")[score].transform(
        lambda s: s / s.max() if s.max() > 0 else np.zeros(len(s))
    )
    SCORE_SENS[f"{score}_rank"] = SCORE_SENS.groupby("region")[score].rank(
        method="min", ascending=False
    )

fig = make_subplots(
    rows=1, cols=5, shared_xaxes=True, shared_yaxes=True,
    horizontal_spacing=.035, subplot_titles=REGION_ORDER,
)

for heading in fig.layout.annotations:
    heading.font = dict(family="Arial", size=20, color=INK)

selection_rows = []

for col, region in enumerate(REGION_ORDER, start=1):
    q = SCORE_SENS[SCORE_SENS.region.eq(region)].copy()
    rho = q[["score_Nr_rank", "score_Nr2_rank"]].corr(
        method="spearman"
    ).iloc[0, 1]

    winners = {
        "N × r": q.sort_values(
            ["score_Nr", "N_pct", "r", "tau", "mask"],
            ascending=[False, False, False, True, True],
        ).iloc[0],
        "N × r²": q.sort_values(
            ["score_Nr2", "N_pct", "r", "tau", "mask"],
            ascending=[False, False, False, True, True],
        ).iloc[0],
    }

    # Background configuration cloud.
    for code in ORDERED_MASKS:
        m = q[q["mask"].eq(code)]

        fig.add_trace(
            go.Scatter(
                x=m.score_Nr_norm, y=m.score_Nr2_norm,
                mode="markers",
                name=f"{code}: {GROUP_LOOKUP[code]['label']}",
                legendgroup=code, showlegend=(col == 1),
                marker=dict(
                    color=MASK_COLORS[code], size=7, opacity=.58,
                    line=dict(color=INK, width=.4),
                ),
                customdata=np.column_stack([
                    m.tau, m.r, m.N_pct,
                    m.score_Nr, m.score_Nr2,
                    m.score_Nr_rank, m.score_Nr2_rank,
                ]),
                hovertemplate=(
                    f"{region} · {code}<br>"
                    "τ=%{customdata[0]:.0f}%"
                    "<br>r=%{customdata[1]:.3f}"
                    " · N=%{customdata[2]:.1f}%"
                    "<br>N×r=%{customdata[3]:.4f}"
                    " · rank %{customdata[5]:.0f}"
                    "<br>N×r²=%{customdata[4]:.4f}"
                    " · rank %{customdata[6]:.0f}"
                    "<extra></extra>"
                ),
            ),
            row=1, col=col,
        )

    winner_styles = {
        "N × r": dict(symbol="star", size=19, color="#D65A4A"),
        "N × r²": dict(symbol="diamond-open", size=21, color=INK),
    }

    for rule, winner in winners.items():
        fig.add_trace(
            go.Scatter(
                x=[winner.score_Nr_norm],
                y=[winner.score_Nr2_norm],
                mode="markers",
                name=f"Selected by {rule}",
                legendgroup=f"selected_{rule}",
                showlegend=(col == 1),
                marker={
                    **winner_styles[rule],
                    "line": dict(color="black", width=1.5),
                },
                hovertemplate=(
                    f"{region}<br>{rule} winner"
                    f"<br>{winner['mask']}: {winner['mask_label']}"
                    f"<br>τ={winner.tau:.0f}%"
                    f"<br>r={winner.r:.3f}"
                    f" · N={winner.N_pct:.1f}%"
                    "<extra></extra>"
                ),
            ),
            row=1, col=col,
        )

        score_col = "score_Nr" if rule == "N × r" else "score_Nr2"
        selection_rows.append({
            "region": region,
            "rule": rule,
            "mask": winner["mask"],
            "mask_label": winner["mask_label"],
            "tau": winner["tau"],
            "r": winner["r"],
            "N_pct": winner["N_pct"],
            "score": winner[score_col],
            "rank_rho": rho,
        })

    # One-to-one reference.
    fig.add_shape(
        type="line", x0=0, y0=0, x1=1, y1=1,
        line=dict(color="#A9B9C9", width=1.2, dash="dot"),
        row=1, col=col,
    )

    axis = "" if col == 1 else str(col)
    winner_Nr, winner_Nr2 = winners["N × r"], winners["N × r²"]

    # Scientifically useful annotation: actual winners.
    fig.add_annotation(
        x=.035, y=.965,
        xref=f"x{axis} domain", yref=f"y{axis} domain",
        text=(
            f"Rank ρ = {rho:.2f}"
            f"<br>N×r: {winner_Nr['mask']} · τ={winner_Nr.tau:.0f}%"
            f"<br>N×r²: {winner_Nr2['mask']} · τ={winner_Nr2.tau:.0f}%"
        ),
        showarrow=False, xanchor="left", yanchor="top", align="left",
        bgcolor="rgba(255,255,255,.90)",
        bordercolor="#A9B9C9", borderwidth=.8, borderpad=5,
        font=dict(family="Arial", size=13, color=INK),
    )

    fig.update_xaxes(
        range=[0, 1.03], tickvals=[0, .25, .5, .75, 1],
        title_text="Relative N × r score",
        title_font=dict(family="Arial", size=16, color=INK),
        tickfont=dict(family="Arial", size=13, color=INK),
        gridcolor=GRID, showline=True, mirror=True,
        linecolor=INK, linewidth=1.3, ticks="outside",
        row=1, col=col,
    )

    fig.update_yaxes(
        range=[0, 1.03], tickvals=[0, .25, .5, .75, 1],
        title_text="Relative N × r² score" if col == 1 else None,
        title_font=dict(family="Arial", size=17, color=INK),
        tickfont=dict(family="Arial", size=13, color=INK),
        gridcolor=GRID, showline=True, mirror=True,
        linecolor=INK, linewidth=1.3, ticks="outside",
        row=1, col=col,
    )

fig.update_layout(
    template="plotly_white",
    width=1800, height=450,
    title=None,
    legend=dict(
        x=1.01, xanchor="left", y=1, yanchor="top",
        font=dict(family="Arial", size=13, color=INK),
    ),
    margin=dict(l=85, r=275, t=60, b=90),
    font=dict(family="Arial", color=INK, size=15),
    paper_bgcolor="white", plot_bgcolor="white",
)

fig.show()

SUPP_SCORE_COMPARISON = export_figure(fig, (FIG_SUPP) / ("section2_composite_score_Nr_vs_Nr2"), width=1800, height=450)

SCORE_SELECTIONS = pd.DataFrame(selection_rows)

display(SCORE_SELECTIONS.round({
    "tau": 0, "r": 3, "N_pct": 1,
    "score": 4, "rank_rho": 3,
}))

SCORE_SELECTIONS.to_csv(
    TABLE_SUPP / "section2_composite_score_Nr_vs_Nr2_selections.csv",
    index=False,
)

## 5. Additional sensitivity diagnostics

The remaining workflows reproduce the earlier support-rule, window and historical-weight experiments. Enable RUN_ADDITIONAL_DIAGNOSTICS=True in setup to run them. They write to diagnostic figure and table folders. The main revision sequence above reuses the existing sensitivity results.

## Data contract

The study clock is **19 January 2012–31 December 2024** throughout. The opening direct and Gap-Filled signals come from the regional CSV, `Region VIII_NTL_VNP46A2.csv`, using `DNB_BRDF_Corrected_NTL` and `Gap_Filled_DNB_BRDF_Corrected_NTL`. The notebook searches `datasets/` for this file, starting with `datasets/vnp46a2/region8_daily_stats/`. Thus the raw–filled comparison has the same dates and exported regional footprint. The RQ1 direct regional `EasternVisayas_DNBBRDF_daily_stats.csv` supplies valid-pixel support and the 1–30-day sensitivity; eight GHSL masks and 1 AM NGCP load follow RQ1 Notebooks 03–04. No RQ2 data are loaded.

All primary NTL metrics use the RQ1-style **0th–95th percentile range**: the upper 5% is capped at the 95th percentile within each signal; uncapped columns are retained for audit. This cap is stated in every relevant figure. The four-day signal is a **non-overlapping mean of the companion CSV's direct regional daily series**, anchored to Haiyan as in the sample RQ2 notebook. RQ2 uses a pixel-first four-day mean and a recovery baseline; those cannot be exactly reconstructed from the RQ1 daily summary CSV. This is an explicitly labelled Román-inspired temporal analogue, not a pixel-first replication. Raw and Gap-Filled $n$ count paired **days**; four-day $n$ counts complete calendar **blocks**. All RMSE values use paired z-scores, so $\mathrm{RMSE}^2=2(1-r)$.


In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    import re, math





    pd.options.display.max_columns = 100
    DATA = ROOT / "datasets"

    def find_region_viii_csv():
        name = "Region VIII_NTL_VNP46A2.csv"
        for directory in (DATA / "vnp46a2" / "region8_daily_stats", DATA / "vnp46a2", DATA):
            path = directory / name
            if path.is_file():
                return path
        match = next((path for path in sorted(DATA.rglob(name)) if path.is_file()), None)
        if match is None:
            raise FileNotFoundError(f"{name} was not found under {DATA.relative_to(PROJECT_ROOT)}")
        return match


    REGION_VIII_CSV = find_region_viii_csv()
    REGIONAL_CSV = DATA / 'vnp46a2/region8_daily_stats/EasternVisayas_DNBBRDF_daily_stats.csv'
    GHSL_DIR = DATA / 'vnp46a2/ghsl_masked_ntl_samar_leyte'
    CROSS_DIR = DATA / 'vnp46a2/visayas_ghsl_masked_daily_stats'
    NGCP_CSV = DATA / 'ngcp/NGCP_Hourly_Demand.csv'
    NGCP_XLSX = DATA / 'ngcp/NGCP_hourly_load.xlsx'
    OUT = TABLE_DIAGNOSTIC
    OUT.mkdir(parents=True, exist_ok=True)

    START, END = pd.Timestamp('2012-01-19'), pd.Timestamp('2024-12-31')
    CALENDAR = pd.date_range(START, END, freq='D', name='date')
    HAIYAN = pd.Timestamp('2013-11-08')
    WINDOWS = list(range(1, 31))
    TAUS = [0, 5, 10, 20, 30, 50, 70, 90]
    SC_SWEEP_TAUS = list(range(101))  # dense 0–100% SC plot, following RQ1 Notebook 03
    TRANSFER_TAUS = [0, 5, 10, 20, 30, 50, 70]
    NTL_UPPER_PERCENTILE = 0.95  # same upper-tail cap for direct and Gap-Filled NTL
    MIN_WINDOW_FRACTION = 0.50  # only for the separately labelled window-supported scenario
    PLATEAU_DELTA_R = 0.02       # sensitivity is repeated below for 0.01 and 0.05
    MIN_DIRECT_COVERAGE = 0.20  # eligibility rule for transfer training islands
    MIN_PAIRS = 5
    GROUPS = {
        '30': ('G1', 'Urban Core', '#D65A4A'),
        '23-30': ('G2', 'Dense Urban', '#A87000'),
        '22-30': ('G3', 'Core & Inner', '#E6A23C'),
        '21-30': ('G4', 'Urban Fringe', '#D9B900'),
        '13-30': ('G5', 'Built-Up / Rural Mix', '#375623'),
        '12-30': ('G6', 'Low–Mid Rural', '#7BAF54'),
        '11-30': ('G7', 'Rural Settlements', '#A8CE74'),
        '10-30': ('G8', 'All Land GHSL classes', '#4E79A7'),
    }
    FULL_LABEL = 'Study area (unmasked)'
    MASK_ORDER = [v[0] for v in GROUPS.values()] + [FULL_LABEL]  # G8 beside unmasked area
    COLOR = {FULL_LABEL: '#183A63', **{v[0]: v[2] for v in GROUPS.values()}}
    print('Same-footprint Region VIII raw + Gap-Filled CSV:', REGION_VIII_CSV.relative_to(PROJECT_ROOT))
    print('Long-record full-area input:', REGIONAL_CSV.relative_to(PROJECT_ROOT))
    print('Output folder:', OUT.relative_to(PROJECT_ROOT))


    def date_from_frame(frame):
        d = frame.copy()
        if 'date' not in d:
            if 'system:index' not in d:
                raise ValueError('Expected date or system:index')
            d['date'] = d['system:index'].astype(str).str.extract(r'(\d{4}_\d{2}_\d{2})')[0].str.replace('_','-',regex=False)
        d['date'] = pd.to_datetime(d['date'], errors='coerce').dt.normalize()
        if d['date'].isna().any() or d['date'].duplicated().any():
            raise ValueError('Missing or duplicate dates in an NTL input')
        return d.set_index('date').sort_index()

    def ngcp_hour1(path):
        if not path.is_file():
            raise FileNotFoundError(path.relative_to(PROJECT_ROOT))
        f = pd.read_csv(path, skiprows=2)
        if f.shape[1] < 4:
            raise ValueError('NGCP CSV layout differs from RQ1 Notebook 03')
        f.columns = ['Date','Coincident Peak','Hour No'] + [str(i) for i in range(1, f.shape[1]-2)]
        f['Date'] = pd.to_datetime(f['Date'], dayfirst=True, errors='coerce').dt.normalize()
        f['1'] = pd.to_numeric(f['1'], errors='coerce')
        f = f.dropna(subset=['Date']).set_index('Date').sort_index()
        if f.index.duplicated().any():
            raise ValueError('Duplicate NGCP dates')
        return f['1'].reindex(CALENDAR).rename('ngcp_1am_mw')

    LOAD = ngcp_hour1(NGCP_CSV)

    def paired_metrics(signal, target, eligible_index=None):
        x, y = pd.Series(signal, dtype=float), pd.Series(target, dtype=float)
        if eligible_index is None:
            eligible_index = y.index[y.notna()]
        eligible_index = pd.Index(eligible_index)
        p = pd.concat([x.rename('ntl'), y.rename('load')], axis=1).reindex(eligible_index).dropna()
        n, possible = len(p), len(eligible_index)
        result = {'n_pairs': n, 'n_possible': possible,
                  'n_pct': 100*n/possible if possible else np.nan,
                  'r': np.nan, 'rmse_z': np.nan}
        if n < MIN_PAIRS or p.ntl.std(ddof=0) == 0 or p.load.std(ddof=0) == 0:
            return result
        a = (p.ntl-p.ntl.mean()) / p.ntl.std(ddof=0)
        b = (p.load-p.load.mean()) / p.load.std(ddof=0)
        result['r'] = float(a.corr(b))
        result['rmse_z'] = float(np.sqrt(np.mean((a-b)**2)))
        return result

## 1. The three full-period signals before reliability qualification

**Purpose:** compare direct DNB-BRDF, its Gap-Filled companion, and a four-day direct composite for all of 2012–2024. The first two are columns from the **same Region VIII export**. The charts let us test whether direct radiance is erratic, whether apparent Gap-Filled continuity occurs when direct observations are sparse, and whether four-day aggregation improves load alignment. These are questions for the results, not conclusions assumed in advance.


In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    companion=date_from_frame(pd.read_csv(REGION_VIII_CSV))
    required={'DNB_BRDF_Corrected_NTL','Gap_Filled_DNB_BRDF_Corrected_NTL'}
    missing=required-set(companion.columns)
    if missing:
        raise ValueError(f'{REGION_VIII_CSV.name}: missing {sorted(missing)}')
    DIRECT_UNCAPPED=pd.to_numeric(companion['DNB_BRDF_Corrected_NTL'],errors='coerce').reindex(CALENDAR)
    GAP_UNCAPPED=pd.to_numeric(companion['Gap_Filled_DNB_BRDF_Corrected_NTL'],errors='coerce').reindex(CALENDAR)
    if DIRECT_UNCAPPED.notna().sum()<365 or GAP_UNCAPPED.notna().sum()<365:
        raise ValueError('The Region VIII CSV must contain the long RQ1 daily record, not an event-only subset.')
    def cap_0_95(s):
        valid=s.dropna()
        return s.clip(lower=valid.quantile(0),upper=valid.quantile(NTL_UPPER_PERCENTILE))
    DIRECT_RAW=cap_0_95(DIRECT_UNCAPPED)
    GAP_RAW=cap_0_95(GAP_UNCAPPED)
    print('NTL P95 caps: direct',round(DIRECT_UNCAPPED.quantile(.95),4),
          'Gap-Filled',round(GAP_UNCAPPED.quantile(.95),4))
    GAP_AVAILABLE=True
    # SC is supplied by RQ1 regional direct statistics, not inferred from filled values.
    direct_frame=date_from_frame(pd.read_csv(REGIONAL_CSV))
    if not {'NTL_mean','Valid_px'} <= set(direct_frame.columns):
        raise ValueError('RQ1 regional CSV must have NTL_mean and Valid_px')
    DIRECT_PX=pd.to_numeric(direct_frame.Valid_px,errors='coerce').reindex(CALENDAR)
    DIRECT_SC=100*DIRECT_PX/DIRECT_PX.max()

    # Event-anchored, non-overlapping, complete four-calendar-day units.
    block_id=np.floor_divide((CALENDAR-HAIYAN).days,4)
    block_start=pd.DatetimeIndex(HAIYAN+pd.to_timedelta(np.unique(block_id)*4,unit='D'))
    block_table=pd.DataFrame({'direct':DIRECT_RAW.to_numpy(),'gap':GAP_RAW.to_numpy(),
                              'load':LOAD.to_numpy(),'block':block_id},index=CALENDAR)
    agg=block_table.groupby('block').agg(direct_mean=('direct','mean'),gap_mean=('gap','mean'),
                                          load_mean=('load','mean'),direct_days=('direct','count'),
                                          gap_days=('gap','count'),load_days=('load','count'),calendar_days=('load','size'))
    agg.index=block_start
    agg=agg[(agg.calendar_days==4)&(agg.load_days==4)]
    agg['four_day_direct']=agg.direct_mean.where(agg.direct_days>0)
    agg.to_csv(OUT/'opening_fullperiod_four_day_block_support.csv')
    ELIGIBLE_DAYS=CALENDAR[LOAD.notna()]
    ELIGIBLE_BLOCKS=agg.index
    OPENING=pd.DataFrame([
     {'signal':'Direct DNB-BRDF (P95)','unit':'days',**paired_metrics(DIRECT_RAW,LOAD,ELIGIBLE_DAYS)},
     {'signal':'Gap-Filled (P95)','unit':'days',**paired_metrics(GAP_RAW,LOAD,ELIGIBLE_DAYS)},
     {'signal':'4-day direct mean (P95)','unit':'4-day blocks',
      **paired_metrics(agg.four_day_direct,agg.load_mean,ELIGIBLE_BLOCKS)}])
    OPENING.to_csv(OUT/'opening_2012_2024_three_signal_metrics.csv',index=False)
    print('Study calendar:',CALENDAR.min().date(),'to',CALENDAR.max().date())
    print('Direct + Gap-Filled source:',REGION_VIII_CSV.relative_to(PROJECT_ROOT))
    display(OPENING.round(3))

### Figure 1A — $r$, RMSE and N%, 2012–2024; NTL capped at P95

The first two denominators are load-available days; the third is complete four-day blocks with all four NGCP days. The metrics use their native clocks, so compare the broad pattern and availability rather than treating a daily $n$ as equivalent to a block $n$. For a common-clock diagnostic, the CSV below also aggregates raw and Gap-Filled daily regional values to the same four-day blocks.

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    common=pd.DataFrame([
     {'signal':'Direct on common 4-day blocks',**paired_metrics(agg.direct_mean,agg.load_mean,ELIGIBLE_BLOCKS)},
     {'signal':'Gap-Filled on common 4-day blocks',**paired_metrics(agg.gap_mean,agg.load_mean,ELIGIBLE_BLOCKS)},
     {'signal':'Four-day direct regional composite',**paired_metrics(agg.four_day_direct,agg.load_mean,ELIGIBLE_BLOCKS)}])
    common.to_csv(OUT/'supplement_fullperiod_common_four_day_clock.csv',index=False)
    if GAP_AVAILABLE:
        colors=['#4E79A7','#D65A4A','#375623']
        fig=make_subplots(rows=1,cols=3,subplot_titles=['Pearson r','Paired z-score RMSE','Retained analysis units (%)'])
        for col,field in enumerate(['r','rmse_z','n_pct'],1):
            vals=OPENING[field].to_numpy(float)
            labels=[f'{int(x.n_pairs)}/{int(x.n_possible)} {x.unit}' for x in OPENING.itertuples()]
            fig.add_trace(go.Bar(x=OPENING.signal,y=vals,marker_color=colors,showlegend=False,
                                 text=labels if field=='n_pct' else [f'{v:.2f}' if np.isfinite(v) else 'undefined' for v in vals],
                                 textposition='outside'),row=1,col=col)
        fig.update_yaxes(title_text='r',row=1,col=1)
        fig.update_yaxes(title_text='RMSE (paired SD units)',row=1,col=2)
        fig.update_yaxes(title_text='n / eligible (%)',range=[0,115],row=1,col=3)
        fig.update_xaxes(tickangle=-18)
        fig.update_layout(template='plotly_white',width=1370,height=500,
                          title='Study area, 2012–2024: P95-capped NTL alignment and N%',
                          margin=dict(l=65,r=30,t=85,b=145),font=dict(color='#233B60'))
        export_figure((fig).update_layout(template='plotly_white', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='#233B60', size=13)), FIG_DIAGNOSTIC / ('fig_01a_fullperiod_three_signal_metrics'), scale=2)

### Figure 1B — The full period first; storm windows are zooms

The full 2012–2024 plot keeps the three P95-capped NTL signals in context. The top strip is directly valid-pixel support from the raw band; high Gap-Filled availability is not new direct evidence. The four-day value is repeated only as a **step display** across its own block, never counted as four independent observations. Haiyan and Rai zooms use the same underlying series and are illustrative, not separate validation samples.

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    # Inputs expected by the replacement Figure 1 plotting block.
    required = [
        "CALENDAR", "HAIYAN", "GHSL_DIR", "date_from_frame",
        "DIRECT_UNCAPPED", "GAP_UNCAPPED", "DIRECT_RAW",
        "DIRECT_SC", "LOAD", "agg", "COLOR", "save_fig",
    ]
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError(
            f"Run Notebook 5's import and initial data-preparation cells first: {missing}"
        )

    FULL_LEFT = pd.Timestamp("2013-01-01")
    PLOT_RIGHT = pd.Timestamp("2023-12-31")
    ZOOM_LEFT = pd.Timestamp("2013-05-01")
    ZOOM_RIGHT = pd.Timestamp("2014-05-31")

    DIRECT = DIRECT_UNCAPPED.copy()
    FILLED = GAP_UNCAPPED.copy()
    NGCP = LOAD.copy()
    AREA_SC = DIRECT_SC.copy()
    NGCP_4D = agg["load_mean"].copy()

    # G3 is GHSL classes 22–30 in Notebook 5.
    g3_path = GHSL_DIR / "EasternVisayas_DNBBRDF_22-30_stats.csv"
    if not g3_path.is_file():
        raise FileNotFoundError(g3_path.relative_to(PROJECT_ROOT))

    g3 = date_from_frame(pd.read_csv(g3_path)).reindex(CALENDAR)
    needed_columns = {"Mean_by_sum", "Valid_px"}
    if not needed_columns <= set(g3.columns):
        raise ValueError(f"{g3_path.name}: missing {needed_columns - set(g3.columns)}")

    g3_ntl = pd.to_numeric(g3["Mean_by_sum"], errors="coerce")
    g3_px = pd.to_numeric(g3["Valid_px"], errors="coerce")
    g3_valid = g3_ntl.notna() & np.isfinite(g3_ntl) & g3_px.gt(0)

    if not g3_valid.any():
        raise ValueError("G3 has no valid direct observations.")

    g3_pixel_denominator = float(g3_px.where(g3_valid).max())
    G3_SC = 100 * g3_px / g3_pixel_denominator

    # The requested RQ signal: P95-capped G3 direct observations,
    # admitted at SC >= 10%, then aggregated by four-day median.
    g3_p95 = float(g3_ntl.where(g3_valid).quantile(0.95))
    g3_qualified = g3_ntl.clip(upper=g3_p95).where(
        g3_valid & G3_SC.ge(10)
    )

    # Use Notebook 5's existing Haiyan-anchored four-day clock.
    block_id = np.floor_divide((CALENDAR - HAIYAN).days, 4)
    block_start = HAIYAN + pd.to_timedelta(block_id * 4, unit="D")

    four_day_inputs = pd.DataFrame(
        {
            "area": DIRECT_RAW.to_numpy(dtype=float),
            "g3": g3_qualified.to_numpy(dtype=float),
            "block_start": block_start,
        },
        index=CALENDAR,
    )
    four_day_medians = four_day_inputs.groupby("block_start")[["area", "g3"]].median()

    DIRECT_4D = four_day_medians["area"].reindex(NGCP_4D.index)
    RQ_4D = four_day_medians["g3"].reindex(NGCP_4D.index)

    # Notebook 5's COLOR already contains G1–G8 colors, but lacks these keys.
    COLOR.update({
        "direct": "#2952CC",
        "filled": "#D65A4A",
        "load": "#1A1A1A",
    })

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    # Keep the preceding data preparation and metrics-table code.
    PLOT_RIGHT = pd.Timestamp("2023-12-31")

    def p95_for_display(series):
        observed = series.loc[FULL_LEFT:PLOT_RIGHT].dropna()
        return series.clip(upper=observed.quantile(.95))

    PLOT_SERIES = {
        'direct': p95_for_display(DIRECT),
        'filled': p95_for_display(FILLED),
        'area4': p95_for_display(DIRECT_4D),
        'rq4': p95_for_display(RQ_4D),
    }
    PLOT_LIMIT = {
        2: 1.07 * max(PLOT_SERIES['direct'].max(),
                      PLOT_SERIES['filled'].max()),
        4: 1.07 * max(PLOT_SERIES['area4'].max(),
                      PLOT_SERIES['rq4'].max()),
    }
    AREA_COLOR = '#2952CC'
    RQ_COLOR = '#0B7A3E'

    def panel_z(series, left, right):
        visible = series.loc[left:right]
        sd = visible.std(ddof=0)
        if not np.isfinite(sd) or sd == 0:
            return series * np.nan
        return (series - visible.mean()) / sd

    def clean_sc_strip(fig, row, col, dates, label, values, colorbar=False):
        support = values.reindex(dates)
        hover = [
            'Missing' if pd.isna(value) else f'{value:.0f}%'
            for value in support
        ]

        fig.add_trace(
            go.Heatmap(
                x=dates,
                y=[label],
                z=[support.fillna(-1).to_numpy(float)],
                customdata=[hover],
                zmin=-1,
                zmax=100,
                colorscale=[
                    [0, '#E6E6E6'],
                    [.0098, '#E6E6E6'],
                    [.0099, '#F7FCF5'],
                    [.25, '#C7E9C0'],
                    [.5, '#74C476'],
                    [.75, '#238B45'],
                    [1, '#00441B'],
                ],
                showscale=colorbar,
                colorbar=dict(
                    title='SC %',
                    x=.958,
                    xanchor='left',
                    y=.50,
                    len=.15,
                    thickness=12,
                    xpad=0,
                    tickvals=[0, 50, 100],
                ),
                hovertemplate=(
                    '%{y}<br>%{x|%d %b %Y}<br>'
                    'SC: %{customdata}<extra></extra>'
                ),
            ),
            row=row,
            col=col,
        )
        fig.update_xaxes(showticklabels=False, row=row, col=col)

    def clean_signal(
        fig, row, col, clock, plotted, actual,
        label, color, legend_id, is_zoom, view, left, right
    ):
        y = plotted if view == 'radiance' else panel_z(
            plotted, left, right
        )

        if is_zoom:
            style = dict(
                mode='lines',
                line=dict(
                    color=color,
                    width=2.1 if label == 'RQ G3 4D' else 1.6,
                    dash='dot' if label == 'Area 4D' else 'solid',
                    shape='linear' if row == 2 else 'hv',
                ),
            )
        else:
            style = dict(
                mode='markers',
                marker=dict(
                    color=color,
                    size=2.5 if row == 2 else 3,
                    opacity=.35,
                ),
            )

        fig.add_trace(
            go.Scatter(
                x=clock,
                y=y.reindex(clock),
                customdata=actual.reindex(clock),
                name=label,
                legend=legend_id,
                connectgaps=False,
                hovertemplate=(
                    '%{x|%d %b %Y}<br>'
                    'Actual radiance=%{customdata:.2f}'
                    '<extra></extra>'
                ),
                **style,
            ),
            row=row,
            col=col,
            secondary_y=False,
        )

    def clean_main_figure(view):
        fig = make_subplots(
            rows=4,
            cols=2,
            row_heights=[.055, .445, .055, .445],
            column_widths=[.57, .43],
            vertical_spacing=.035,
            horizontal_spacing=.07,
            specs=[
                [{'type': 'heatmap'}, {'type': 'heatmap'}],
                [{'secondary_y': True}, {'secondary_y': True}],
                [{'type': 'heatmap'}, {'type': 'heatmap'}],
                [{'secondary_y': True}, {'secondary_y': True}],
            ],
        )

        for col, left, right in [
            (1, FULL_LEFT, PLOT_RIGHT),
            (2, ZOOM_LEFT, ZOOM_RIGHT),
        ]:
            is_zoom = col == 2
            dates = CALENDAR[(CALENDAR >= left) & (CALENDAR <= right)]
            blocks = NGCP_4D.loc[left:right].index
            top_legend = 'legend' if col == 1 else 'legend2'
            bottom_legend = 'legend3' if col == 1 else 'legend4'

            clean_sc_strip(
                fig, 1, col, dates, 'Area', AREA_SC,
                colorbar=is_zoom,
            )
            clean_sc_strip(
                fig, 3, col, dates, 'G3', G3_SC,
            )

            for plotted, actual, label, color in [
                (PLOT_SERIES['direct'], DIRECT, 'Direct', COLOR['direct']),
                (PLOT_SERIES['filled'], FILLED, 'Gap-Filled', COLOR['filled']),
            ]:
                clean_signal(
                    fig, 2, col, dates, plotted, actual,
                    label, color, top_legend, is_zoom,
                    view, left, right,
                )

            for plotted, actual, label, color in [
                (PLOT_SERIES['area4'], DIRECT_4D,
                 'Area 4D', AREA_COLOR),
                (PLOT_SERIES['rq4'], RQ_4D,
                 'RQ G3 4D', RQ_COLOR),
            ]:
                clean_signal(
                    fig, 4, col, blocks, plotted, actual,
                    label, color, bottom_legend, is_zoom,
                    view, left, right,
                )

            for row, clock, load, label, legend_id in [
                (2, dates, NGCP, 'NGCP 1 AM', top_legend),
                (4, blocks, NGCP_4D, 'NGCP 4D', bottom_legend),
            ]:
                y = (
                    load.reindex(clock)
                    if view == 'radiance'
                    else panel_z(load, left, right).reindex(clock)
                )
                fig.add_trace(
                    go.Scatter(
                        x=clock,
                        y=y,
                        mode='lines',
                        name=label,
                        legend=legend_id,
                        line=dict(
                            color=COLOR['load'],
                            width=1.5,
                            shape='linear' if row == 2 else 'hv',
                        ),
                        connectgaps=False,
                    ),
                    row=row,
                    col=col,
                    secondary_y=(view == 'radiance'),
                )
                if view == 'radiance':
                    fig.update_yaxes(
                        range=[0, PLOT_LIMIT[row]],
                        row=row,
                        col=col,
                        secondary_y=False,
                    )
                fig.add_vline(
                    x=HAIYAN,
                    line_dash='dash',
                    line_color='#233B60',
                    row=row,
                    col=col,
                )

            for row in range(1, 5):
                fig.update_xaxes(range=[left, right], row=row, col=col)
            fig.update_xaxes(showticklabels=True, row=2, col=col)
            fig.update_xaxes(title_text='Date', row=4, col=col)

        for row in (2, 4):
            fig.update_yaxes(
                title_text=(
                    'Radiance (nW·cm⁻²·sr⁻¹)'
                    if view == 'radiance' else 'z-score'
                ),
                row=row,
                col=1,
                secondary_y=False,
            )
            if view == 'radiance':
                fig.update_yaxes(
                    title_text='NGCP load (MW)',
                    row=row,
                    col=2,
                    secondary_y=True,
                )
            else:
                for col in (1, 2):
                    fig.update_yaxes(
                        visible=False,
                        row=row,
                        col=col,
                        secondary_y=True,
                    )

        legends = {
            'legend': dict(x=.53, y=.905),
            'legend2': dict(x=.99, y=.905),
            'legend3': dict(x=.53, y=.39),
            'legend4': dict(x=.99, y=.39),
        }
        for item in legends.values():
            item.update(
                orientation='h',
                xanchor='right',
                yanchor='top',
                bgcolor='rgba(255,255,255,.9)',
                font=dict(size=11),
            )

        fig.update_layout(
            template='plotly_white',
            paper_bgcolor='rgba(0,0,0,0)',
            width=1500,
            height=850,
            margin=dict(l=85, r=70, t=20, b=75),
            font=dict(size=12, color='#233B60'),
            **legends,
        )
        export_figure((fig).update_layout(template='plotly_white', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='#233B60', size=13)), FIG_DIAGNOSTIC / (f'fig_01_2013_2023_clean_{view}'), scale=2)

    clean_main_figure('radiance')
    clean_main_figure('zscore')

**Interpretation gate.** Inspect direct support alongside apparent Gap-Filled flat stretches; the CSV alone does not establish the exact gap-filling mechanism, so do not claim every flat section literally repeats the last observation. The opening direct and Gap-Filled columns are paired from one export. The SC panel comes from the RQ1 regional direct-statistics export and should be checked for footprint consistency before using it as a pixel-level explanation of the companion CSV.


## 2. Bring in the long RQ1 direct record and GHSL masks

**Question.** Does the relationship change as the spatial extent narrows from the **study area without a GHSL mask** to settlement-focused support? RQ1 Notebook 00 provides the unmasked regional direct-band CSV; Notebook 03 provides eight cumulative GHSL-mask CSVs. The unmasked area's `NTL_mean` and GHSL masks' `Mean_by_sum` are compared as standardized trajectories and metrics, not absolute radiance levels. **G8 is not the unmasked study area:** G8 retains GHSL classes 10–30 and excludes pixels outside those classes. Their rows are placed together in the figures so the extent difference is visible.

Following Notebook 03, each direct series requires positive valid-pixel count and uses a **0th–95th percentile range** (upper 5% capped). `SC = 100 × Valid_px / max(Valid_px)` is normalized separately within each spatial support. A common τ therefore means the same fraction of that support's own maximum observed valid pixels, not an equal pixel count across supports. At τ=0, all dates with direct valid pixels remain eligible.


In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    def read_daily_csv(path, value_col):
        if not path.is_file():
            raise FileNotFoundError(path.relative_to(PROJECT_ROOT))
        d = date_from_frame(pd.read_csv(path))
        missing = {value_col,'Valid_px'} - set(d.columns)
        if missing:
            raise ValueError(f'{path.name}: missing {sorted(missing)}')
        d['value'] = pd.to_numeric(d[value_col],errors='coerce')
        d['Valid_px'] = pd.to_numeric(d.Valid_px,errors='coerce')
        return d[['value','Valid_px']].reindex(CALENDAR)

    def prep_direct(d):
        f=d.copy()
        valid=(f.Valid_px>0)&f.value.notna()&np.isfinite(f.value)
        if not valid.any():
            raise ValueError('No direct dates in a spatial scale')
        denominator=float(f.loc[valid,'Valid_px'].max())
        hi=float(f.loc[valid,'value'].quantile(.95))
        f['SC_pct']=100*f.Valid_px/denominator
        f['ntl']=f.value.clip(upper=hi).where(valid)
        return f[['ntl','Valid_px','SC_pct']],{'max_valid_px':denominator,'upper_95_cap':hi,'direct_dates':int(valid.sum())}

    REGION_DAILY=read_daily_csv(REGIONAL_CSV,'NTL_mean')
    SCALES={FULL_LABEL:prep_direct(REGION_DAILY)[0]}
    inventory=[{'scale':FULL_LABEL,'source':REGIONAL_CSV.relative_to(PROJECT_ROOT).as_posix(),'value_col':'NTL_mean',
                **prep_direct(REGION_DAILY)[1]}]
    for group,(mask,label,color) in GROUPS.items():
        path=GHSL_DIR / f'EasternVisayas_DNBBRDF_{group}_stats.csv'
        d=read_daily_csv(path,'Mean_by_sum')
        SCALES[mask],audit=prep_direct(d)
        inventory.append({'scale':mask,'source':path.relative_to(PROJECT_ROOT).as_posix(),'value_col':'Mean_by_sum',**audit})
    INPUT_AUDIT=pd.DataFrame(inventory)
    INPUT_AUDIT.to_csv(OUT/'qualified_input_inventory.csv',index=False)
    print(f'Loaded long direct series for {len(SCALES)} spatial scales; source and denominator audit saved.')

### Figure 2 — The spatial and observational qualification steps

The top panel plots Pearson $r$ for the unsmoothed P95-capped direct series over the **full SC threshold range**; the lower panel shows **N%**, the fraction of load dates with an admitted direct observation. Read each line as a trade-off, not a universal optimum. A GHSL mask changes which pixels contribute; τ changes which dates are admitted within that mask. Because each τ has a different paired-date set, a higher $r$ alone is insufficient evidence of better measurement.


In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    intro_rows=[]
    for scale,d in SCALES.items():
        for tau in SC_SWEEP_TAUS:
            admitted=d.ntl.notna()&(d.SC_pct>=tau)&LOAD.notna()
            m=paired_metrics(d.ntl.where(admitted),LOAD,LOAD.index[LOAD.notna()])
            intro_rows.append({'scale':scale,'tau':tau,'direct_pct':100*admitted.sum()/LOAD.notna().sum(),**m})
    INTRO=pd.DataFrame(intro_rows)
    INTRO.to_csv(OUT/'reliability_conditions_direct_dates.csv',index=False)
    FIG2_SCALES=MASK_ORDER
    fig=make_subplots(rows=2,cols=1,shared_xaxes=True,vertical_spacing=.11,
                      subplot_titles=['Direct NTL–load alignment','Directly paired load dates (N%)'])
    for scale in FIG2_SCALES:
        q=INTRO[INTRO.scale==scale].sort_values('tau')
        for row,field in [(1,'r'),(2,'direct_pct')]:
            fig.add_trace(go.Scatter(x=q.tau,y=q[field],mode='lines',name=scale,
                                     showlegend=(row==1),line=dict(color=COLOR[scale],width=3)),row=row,col=1)
    for row in (1,2):
        for threshold in (10,50):
            fig.add_vline(x=threshold,row=row,col=1,line_dash='dot',line_color='#9CACBD')
    fig.update_xaxes(title_text='SC threshold τ (%)',range=[0,100],dtick=10,row=2,col=1)
    fig.update_yaxes(title_text='Pearson r',row=1,col=1)
    fig.update_yaxes(title_text='N% of load dates',range=[0,100],row=2,col=1)
    fig.update_layout(template='plotly_white',height=730,width=1200,
                      title='SC threshold trade-off after P95 capping: 2012–2024',
                      legend=dict(orientation='v',x=1.02,y=1,font=dict(size=13)),
                      paper_bgcolor='rgba(0,0,0,0)',margin=dict(l=75,r=270,t=90,b=75),
                      font=dict(color='#233B60',size=13))
    export_figure((fig).update_layout(template='plotly_white', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='#233B60', size=13)), FIG_DIAGNOSTIC / ('fig_02_extent_and_sc_introduction'), scale=2)

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    INTRO = pd.DataFrame(intro_rows)
    INTRO.to_csv(
        OUT / 'reliability_conditions_direct_dates_2013_2024.csv',
        index=False,
    )

    # Take the names directly from your existing RQ1 GROUPS definition.
    GROUP_LABELS = {
        group_id: f'{group_id} {description}'
        for group_id, description, _ in GROUPS.values()
    }
    FIG2_SCALES = [f'G{i}' for i in range(1, 9)]
    CHECKPOINTS = (10, 50, 90)

    fig = make_subplots(
        rows=2,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=.09,
        row_heights=[.52, .48],
    )

    for scale in FIG2_SCALES:
        q = INTRO[INTRO.scale == scale].sort_values('tau')
        checkpoints = q[q.tau.isin(CHECKPOINTS)]

        fig.add_trace(
            go.Scatter(
                x=q.tau,
                y=q.r,                 # no added 30-pair cutoff
                mode='lines',
                name=GROUP_LABELS[scale],
                line=dict(color=COLOR[scale], width=2.6),
            ),
            row=1,
            col=1,
        )

        # Checkpoint dots for every G1–G8 correlation curve.
        fig.add_trace(
            go.Scatter(
                x=checkpoints.tau,
                y=checkpoints.r,
                customdata=checkpoints.n_pairs,
                mode='markers',
                showlegend=False,
                marker=dict(
                    color=COLOR[scale],
                    size=7,
                    line=dict(color='white', width=1),
                ),
                hovertemplate=(
                    f'{GROUP_LABELS[scale]}<br>'
                    'τ=%{x}%<br>r=%{y:.2f}<br>'
                    'Paired dates=%{customdata}<extra></extra>'
                ),
            ),
            row=1,
            col=1,
        )

        fig.add_trace(
            go.Scatter(
                x=q.tau,
                y=q.n_pct,
                mode='lines',
                name=GROUP_LABELS[scale],
                showlegend=False,
                line=dict(color=COLOR[scale], width=2.6),
            ),
            row=2,
            col=1,
        )

    # The guides become darker as the SC threshold increases.
    CHECKPOINT_COLORS = {
        10: '#C6D9DF',
        50: '#779EAA',
        90: '#315E6D',
    }
    for tau, guide_color in CHECKPOINT_COLORS.items():
        for row in (1, 2):
            fig.add_vline(
                x=tau,
                row=row,
                col=1,
                line_dash='dash',
                line_width=1.5,
                line_color=guide_color,
            )

    # Label the three representative curves at each checkpoint.
    # All eight curves still have checkpoint dots and full names in the legend.
    BOTTOM_OFFSETS = {
        10: {'G1': 25, 'G5': -18, 'G8': -42},
        50: {'G1': 25, 'G5': -18, 'G8': -42},
        90: {'G1': -18, 'G5': -40, 'G8': -62},
    }

    for tau in CHECKPOINTS:
        for scale in ('G1', 'G5', 'G8'):
            point = INTRO[
                (INTRO.scale == scale) & (INTRO.tau == tau)
            ]
            if point.empty:
                continue

            r = float(point.r.iloc[0])
            retained = float(point.n_pct.iloc[0])

            if np.isfinite(r):
                fig.add_annotation(
                    x=tau,
                    y=r,
                    text=f'{scale} {r:.2f}',
                    showarrow=True,
                    arrowhead=0,
                    arrowwidth=1,
                    arrowcolor=COLOR[scale],
                    ax=0,
                    ay=-19,
                    bgcolor='rgba(255,255,255,.92)',
                    bordercolor=COLOR[scale],
                    borderwidth=.7,
                    borderpad=2,
                    font=dict(color=COLOR[scale], size=10),
                    row=1,
                    col=1,
                )

            fig.add_trace(
                go.Scatter(
                    x=[tau],
                    y=[retained],
                    mode='markers',
                    showlegend=False,
                    marker=dict(
                        color=COLOR[scale],
                        size=8,
                        line=dict(color='white', width=1),
                    ),
                ),
                row=2,
                col=1,
            )
            fig.add_annotation(
                x=tau,
                y=retained,
                text=f'{scale} {retained:.1f}%',
                showarrow=True,
                arrowhead=0,
                arrowwidth=1,
                arrowcolor=COLOR[scale],
                ax=0,
                ay=BOTTOM_OFFSETS[tau][scale],
                bgcolor='rgba(255,255,255,.92)',
                bordercolor=COLOR[scale],
                borderwidth=.7,
                borderpad=2,
                font=dict(color=COLOR[scale], size=10),
                row=2,
                col=1,
            )

    fig.update_xaxes(
        title_text='SC threshold τ (%)',
        range=[0, 100],
        dtick=10,
        row=2,
        col=1,
    )
    fig.update_yaxes(title_text='Pearson r', row=1, col=1)
    fig.update_yaxes(
        title_text='N% retained coverage',
        range=[0, 105],
        row=2,
        col=1,
    )
    fig.update_layout(
        template='plotly_white',
        width=1250,
        height=760,
        title='SC threshold trade-off after P95 capping, 2013–2024',
        legend=dict(
            orientation='v',
            x=1.02,
            y=1,
            font=dict(size=11),
        ),
        paper_bgcolor='rgba(0,0,0,0)',
        margin=dict(l=75, r=265, t=80, b=75),
        font=dict(color='#233B60', size=13),
    )
    export_figure((fig).update_layout(template='plotly_white', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='#233B60', size=13)), FIG_DIAGNOSTIC / ('fig_02_sc_tradeoff_2013_2024'), scale=2)

Discussion: Alignment generally increased as the spatial-completeness threshold rose, while the fraction of retained load dates declined. Because thresholding selects dates rather than modifying their radiance, the increase represents stronger conditional agreement on well-observed dates; it does not establish better performance across the full calendar. Stringent thresholds may exclude precisely the cloud-affected storm periods of interest. We therefore evaluate correlation together with retained coverage and select thresholds by the intended application, rather than maximizing correlation alone.

## 3. Sensitivity cube: spatial scale × SC threshold × 1–30 calendar days

**Question.** What smoothing horizon is enough for a useful long-term signal, and how much support does it require? We compute **both** of the following lenses for every combination:

- **Direct-date lens (primary):** apply SC to daily direct NTL; smooth the admitted values over a centred *calendar* window; evaluate only dates with a directly admitted NTL observation and a load observation. `n%` is therefore normally constant across windows at fixed mask and τ. This makes the trade-off between $r$ and loss of directly observed dates explicit.
- **Window-supported lens (diagnostic):** require at least 50% of the calendar days in each window to contain an SC-admitted direct observation. An estimate can then exist on a date without a direct NTL observation. Its `n%` is **supported-estimate availability**, not directly observed date coverage. We report the median number of contributing direct days and show 25/50/75% support-rule sensitivity later.

The primary retrospective windows are centred, matching the spirit of Notebook 03's retrospective smoothing. A trailing-window check follows for prospective use. Both NTL and NGCP load are smoothed on the same calendar width; consequently rising $r$ describes agreement at a broader time scale, not better daily outage detection. `w=1` is unsmoothed direct NTL. This calendar-day sweep is **not** a reproduction of the original 30-retained-observation method.

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    def smooth_pair(d,load,tau,w,lens='direct',support_fraction=.50,center=True):
        admitted=d.ntl.notna()&(d.Valid_px>0)&(d.SC_pct>=tau)
        x=d.ntl.where(admitted)
        if lens=='complete':
            # Trailing, nested windows: every constituent calendar day must be direct and SC-admitted.
            # This guarantees N%(w+1) <= N%(w), unlike partial windows that can bridge gaps.
            count=x.notna().astype(int).rolling(w,min_periods=w).sum()
            xs=x.rolling(w,min_periods=w).mean()
            ys=load.rolling(w,min_periods=w).mean()
        elif lens in ('direct','window'):
            count=x.notna().astype(int).rolling(w,center=center,min_periods=1).sum()
            needed=1 if lens=='direct' else max(1,math.ceil(support_fraction*w))
            xs=x.rolling(w,center=center,min_periods=needed).mean()
            ys=load.rolling(w,center=center,min_periods=needed).mean()
            if lens=='direct': xs=xs.where(admitted)
        else:
            raise ValueError(f'Unknown support lens: {lens}')
        xs=xs.where(load.notna())
        ys=ys.where(load.notna())
        return xs,ys,admitted,count

    def configuration_metric(d,load,tau,w,lens='direct',support_fraction=.50,center=True):
        x,y,admitted,count=smooth_pair(d,load,tau,w,lens,support_fraction,center)
        m=paired_metrics(x,y,load.index[load.notna()])
        paired=x.notna()&y.notna()
        m.update(direct_n=int((admitted&load.notna()).sum()),
                 direct_pct=100*(admitted&load.notna()).sum()/load.notna().sum(),
                 median_contributing_days=float(count[paired].median()) if paired.any() else np.nan,
                 supported_without_target_direct=int((paired&~admitted).sum()))
        return m

    # Synthetic logic checks only; these are not the RQ1 results.
    toy_i=pd.date_range('2020-01-01',periods=5)
    toy_d=pd.DataFrame({'ntl':[1.,np.nan,np.nan,4.,5.],
                        'Valid_px':[10,0,0,10,10],
                        'SC_pct':[100,0,0,100,100]},index=toy_i)
    toy_l=pd.Series([1.,2.,3.,4.,5.],index=toy_i)
    a,b,ad,c=smooth_pair(toy_d,toy_l,0,3,'direct')
    assert a.isna().iloc[1] and ad.sum()==3
    a,b,ad,c=smooth_pair(toy_d,toy_l,0,3,'window',.5)
    assert c.iloc[3]>=2 and a.iloc[3]==a.iloc[3]
    assert configuration_metric(toy_d,toy_l,0,3,'complete')['n_pairs'] <= configuration_metric(toy_d,toy_l,0,1,'complete')['n_pairs']

    rows=[]
    for scale,d in SCALES.items():
        for tau in TAUS:
            for w in WINDOWS:
                for lens in ['direct','window','complete']:
                    rows.append({'scale':scale,'tau':tau,'window_days':w,'lens':lens,
                                 'window_rule':'trailing complete' if lens=='complete' else 'centered partial',
                                 **configuration_metric(d,LOAD,tau,w,lens,MIN_WINDOW_FRACTION)})
    MATRIX=pd.DataFrame(rows)
    MATRIX.to_csv(OUT/'mask_sc_window_1_to_30_metric_cube.csv',index=False)
    check=MATRIX[MATRIX.lens=='complete'].sort_values('window_days').groupby(['scale','tau']).n_pairs.apply(lambda s:s.diff().dropna().le(0).all())
    assert check.all(), 'Complete-window paired N must not increase with width'
    print(f'Saved {len(MATRIX):,} mask × SC × window × support-rule results.')

### Figure 3 — 1–30-day sensitivity on the same paired-date rule

Each row now uses **complete trailing windows**. At width $w$, the endpoint is scored only if **all $w$ calendar days** have an SC-admitted direct observation and NGCP load. Thus $r$, paired z-score RMSE and **N%** are computed on the same window-qualified endpoints; N% must fall or stay level as $w$ grows. Missing curve segments mean too few valid pairs to estimate $r$, an important limitation rather than a plotting error. This strict rule is deliberately conservative in a cloud-impacted setting.

The exported cube also includes the previous **direct-anchor** rule (center date must be observed) and a **50%-supported partial-window** rule. Their N% can stay flat or rise, respectively; those are different estimands and are shown separately in the supplement. RMSE is mathematically linked to $r$ after paired standardization: $\mathrm{RMSE}^2=2(1-r)$.


In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    WINDOW_CHECKPOINTS = (4, 7, 14, 30)
    ALL_GROUPS = [f'G{i}' for i in range(1, 9)]
    GROUP_LABELS = {
        group_id: f'{group_id} {description}'
        for group_id, description, _ in GROUPS.values()
    }
    METRIC_LABELS = {
        'r': 'Pearson r',
        'rmse_z': 'Paired z-score RMSE',
        'n_pct': 'N% retained coverage',
    }

    def metric_curve_figure(df, metric, scales, taus, lens, name):
        fig = make_subplots(
            rows=1,
            cols=3,
            shared_yaxes=True,
            horizontal_spacing=.055,
            subplot_titles=[f'SC τ={tau}%' for tau in taus],
        )

        x_offset = {4: 18, 7: 32, 14: 0, 30: -32}
        y_offset = {
            4:  {'G1': -18, 'G5': 18,  'G8': 38},
            7:  {'G1': 18,  'G5': -18, 'G8': -38},
            14: {'G1': -18, 'G5': 18,  'G8': 38},
            30: {'G1': 18,  'G5': -18, 'G8': -38},
        }

        for col, tau in enumerate(taus, start=1):
            panel = df[(df.lens == lens) & (df.tau == tau)]

            for day in WINDOW_CHECKPOINTS:
                fig.add_vline(
                    x=day,
                    row=1,
                    col=col,
                    line_dash='dot',
                    line_color='#B6C7D3',
                    line_width=1,
                )

            for scale in scales:
                q = panel[panel.scale == scale].sort_values('window_days')

                fig.add_trace(
                    go.Scatter(
                        x=q.window_days,
                        y=q[metric],
                        mode='lines',
                        name=GROUP_LABELS[scale],
                        showlegend=(col == 1),
                        line=dict(color=COLOR[scale], width=2.2),
                        connectgaps=False,
                    ),
                    row=1,
                    col=col,
                )

            # Match the SC plot's small outlined labels on selected curves.
            for day in WINDOW_CHECKPOINTS:
                for scale in ('G1', 'G5', 'G8'):
                    point = panel[
                        (panel.scale == scale) &
                        (panel.window_days == day)
                    ]
                    if point.empty:
                        continue

                    value = point[metric].iloc[0]
                    if pd.isna(value):
                        continue

                    label = (
                        f'{scale} {value:.1f}%'
                        if metric == 'n_pct'
                        else f'{scale} {value:.2f}'
                    )
                    ay = y_offset[day][scale]

                    # Keep low-coverage labels above the axis.
                    if metric == 'n_pct' and value < 10:
                        ay = {'G1': -18, 'G5': -38, 'G8': -58}[scale]

                    fig.add_trace(
                        go.Scatter(
                            x=[day],
                            y=[value],
                            mode='markers',
                            showlegend=False,
                            marker=dict(
                                color=COLOR[scale],
                                size=6,
                                line=dict(color='white', width=1),
                            ),
                        ),
                        row=1,
                        col=col,
                    )
                    fig.add_annotation(
                        x=day,
                        y=value,
                        text=label,
                        showarrow=True,
                        arrowhead=0,
                        arrowwidth=.8,
                        arrowcolor=COLOR[scale],
                        ax=x_offset[day],
                        ay=ay,
                        bgcolor='rgba(255,255,255,.95)',
                        bordercolor=COLOR[scale],
                        borderwidth=.7,
                        borderpad=2,
                        font=dict(color=COLOR[scale], size=9),
                        row=1,
                        col=col,
                    )

            fig.update_xaxes(
                title_text='Calendar window (days)',
                range=[1, 30.5],
                tickvals=[1, 4, 7, 14, 20, 25, 30],
                row=1,
                col=col,
            )

        fig.update_yaxes(
            title_text=METRIC_LABELS[metric],
            row=1,
            col=1,
        )
        if metric == 'n_pct':
            fig.update_yaxes(range=[0, 100])

        fig.update_layout(
            template='plotly_white',
            width=1650,
            height=580,
            title=name,
            legend=dict(x=1.01, y=1, font=dict(size=11)),
            paper_bgcolor='rgba(0,0,0,0)',
            margin=dict(l=80, r=245, t=85, b=85),
            font=dict(color='#233B60', size=12),
        )
        return fig

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    export_figure((metric_curve_figure(
            MATRIX, 'r', ALL_GROUPS, [0, 10, 50], 'complete',
            'Alignment across calendar windows and SC thresholds',
        )).update_layout(template='plotly_white', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='#233B60', size=13)), FIG_DIAGNOSTIC / ('fig_03a_r_window_curves'), scale=2)

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    export_figure((metric_curve_figure(
            MATRIX, 'rmse_z', ALL_GROUPS, [0, 10, 50], 'complete',
            'Paired standardized error across calendar windows and SC thresholds',
        )).update_layout(template='plotly_white', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='#233B60', size=13)), FIG_DIAGNOSTIC / ('fig_03b_rmse_window_curves'), scale=2)

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    export_figure((metric_curve_figure(
            MATRIX, 'n_pct', ALL_GROUPS, [0, 10, 50], 'complete',
            'Retained coverage across calendar windows and SC thresholds',
        )).update_layout(template='plotly_white', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='#233B60', size=13)), FIG_DIAGNOSTIC / ('fig_03c_n_window_curves'), scale=2)

### Figure 4 — SC and temporal support are separate dimensions

**Figure 4A** colors direct **N%** by spatial support and SC threshold. **Figures 4B–C** put 1–30 days on the horizontal axis; their color is complete-window **N%** or $r$ on the *same eligible endpoints*. G8 sits beside the unmasked study area, making their difference easy to inspect. A 50%-supported window heatmap is retained as a supplement to show how partial windows can create more estimates when widened; those estimates are not complete direct observation. SC is a spatial fraction, normalized within each mask, and does not itself vary with temporal width.


In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    direct_heat=INTRO.pivot(index='scale',columns='tau',values='direct_pct').reindex(MASK_ORDER)
    fig=go.Figure(go.Heatmap(x=direct_heat.columns,y=direct_heat.index,z=direct_heat.values,
                             colorscale='Blues',zmin=0,zmax=100,colorbar=dict(title='N%'),
                             hovertemplate='Scale %{y}<br>SC τ=%{x}%<br>Direct dates=%{z:.1f}%<extra></extra>'))
    fig.update_layout(template='plotly_white',width=1080,height=560,
                      title='Direct N% by spatial support and SC threshold',
                      xaxis_title='SC threshold τ (%)',yaxis_title='Spatial scale',
                      margin=dict(l=190,r=35,t=75,b=70),font=dict(color='#233B60'))
    export_figure((fig).update_layout(template='plotly_white', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='#233B60', size=13)), FIG_DIAGNOSTIC / ('fig_04a_direct_retention_sc_heatmap'), scale=2)

    def window_heatmap(metric,lens,taus,name,title):
        f=make_subplots(rows=1,cols=len(taus),subplot_titles=[f'SC τ={t}%' for t in taus],
                        shared_yaxes=True,horizontal_spacing=.035)
        for col,tau in enumerate(taus,start=1):
            q=MATRIX[(MATRIX.lens==lens)&(MATRIX.tau==tau)]
            p=q.pivot(index='scale',columns='window_days',values=metric).reindex(MASK_ORDER)
            f.add_trace(go.Heatmap(x=p.columns,y=p.index,z=p.values,colorscale='Blues',
                                   zmin=0 if metric=='n_pct' else float(MATRIX[(MATRIX.lens==lens)&(MATRIX.tau.isin(taus))][metric].min()),
                                   zmax=100 if metric=='n_pct' else 1,
                                   showscale=(col==len(taus)),
                                   colorbar=dict(title='N%' if metric=='n_pct' else 'Pearson r'),
                                   hovertemplate='Scale %{y}<br>Window %{x} d<br>Value %{z:.2f}<extra></extra>'),row=1,col=col)
        f.update_xaxes(title_text='Calendar window (days)',dtick=5)
        f.update_yaxes(title_text='Spatial scale',row=1,col=1)
        f.update_layout(template='plotly_white',width=1450,height=570,title=title,
                        margin=dict(l=190,r=75,t=75,b=75),font=dict(color='#233B60'))
        export_figure((f).update_layout(template='plotly_white', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='#233B60', size=13)), FIG_DIAGNOSTIC / (name), scale=2)

    window_heatmap('n_pct','complete',[0,10,50],'fig_04b_complete_n_mask_window_heatmap',
                   'Complete-window N%: spatial support × width, by SC threshold')
    window_heatmap('r','complete',[0,10,50],'fig_04c_complete_r_mask_window_heatmap',
                   'Complete-window r: same settings and paired endpoints')
    window_heatmap('n_pct','window',[0,10,50],'fig_s_partial_window_n_heatmap',
                   'Supplement: N% with at least 50% direct support in each window')

## 4. Choosing a window and testing composite-score weights

**Step 1: preserve temporal resolution.** For each GHSL mask × τ, the notebook selects the **shortest** 1–30-day direct-anchor window within Δ$r=0.02$ of that setting's best $r$. The result is repeated at Δ$r=0.01$ and 0.05. This is a declared practical tolerance, not a significance test. The direct-anchor score is the manuscript-comparable branch: its **N% reflects admitted direct dates and does not depend on window width**; width is handled by the near-plateau rule. The complete-window curves above quantify the stricter temporal-support cost.

**Step 2: make preferences explicit.** The primary two-axis score varies alignment weight $\lambda$ from 0 to 1:

$$S_\lambda=\lambda(r+1)/2+(1-\lambda)N_{\rm direct}/100.$$

This asks how much selection changes when an application values load alignment over directly observed temporal coverage. Paired standardized RMSE is linked exactly to $r$ by $\mathrm{RMSE}^2=2(1-r)$, so counting both as independent evidence would double-weight alignment.

**Step 3: stress-test the manuscript's three weights.** We also sweep **all 231 weight triples** in 0.05 increments that sum to one across $r$, inverse paired-z RMSE, and N%. Fixed 0–1 scales are used: $(r+1)/2$, $1-\mathrm{RMSE}/2$, and $N/100$. The manuscript's **0.6/0.1/0.3** point is marked. The fixed scaling makes this a sensitivity audit of those weights, not a numerical reproduction of Table 4's min–max normalization. The winner map and top-five frequency show whether that choice sits within a stable region. Since $r$ and RMSE are redundant, this three-weight sweep is a *robustness audit of the historical score*, while the two-axis score is the cleaner decision rule.


In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    def plateau_candidates(frame,delta,group_keys=['scale','tau']):
        candidates=[]
        source=frame[(frame.lens=='direct')&frame.r.notna()&
                     (frame.n_pct>=100*MIN_DIRECT_COVERAGE)&(frame.n_pairs>=30)].copy()
        for _,g in source.groupby(group_keys,sort=False):
            best=g.r.max()
            pick=g[g.r>=best-delta].sort_values('window_days').iloc[0]
            candidates.append(pick)
        return pd.DataFrame(candidates).reset_index(drop=True) if candidates else pd.DataFrame()

    LOCAL_CANDIDATES=plateau_candidates(MATRIX,PLATEAU_DELTA_R)
    LOCAL_CANDIDATES=LOCAL_CANDIDATES[LOCAL_CANDIDATES.scale!=FULL_LABEL].copy()
    if LOCAL_CANDIDATES.empty:
        raise ValueError('No locally eligible GHSL candidates: inspect n, SC and minimum coverage')
    LOCAL_CANDIDATES.to_csv(OUT/'local_shortest_near_plateau_candidates.csv',index=False)

    def primary_score(frame,lam):
        q=frame.copy()
        q['score']=lam*(q.r+1)/2+(1-lam)*q.n_pct/100
        return q.sort_values(['score','n_pct','window_days','scale','tau'],
                             ascending=[False,False,True,True,True]).reset_index(drop=True)

    LOCAL_WEIGHTS=[]
    for lam in np.round(np.arange(0,1.0001,.05),2):
        winner=primary_score(LOCAL_CANDIDATES,lam).iloc[0]
        LOCAL_WEIGHTS.append({'lambda_alignment':float(lam),'selected_scale':winner.scale,
                              'selected_tau':int(winner.tau),'selected_window_days':int(winner.window_days),
                              'r':winner.r,'rmse_z':winner.rmse_z,'n_pairs':int(winner.n_pairs),
                              'n_pct':winner.n_pct,'score':winner.score})
    LOCAL_WEIGHTS=pd.DataFrame(LOCAL_WEIGHTS)
    LOCAL_WEIGHTS.to_csv(OUT/'local_weight_sweep_selected_configurations.csv',index=False)

    def minmax(s):
        span=s.max()-s.min()
        return pd.Series(.5,index=s.index) if not np.isfinite(span) or span==0 else (s-s.min())/span
    historic=LOCAL_CANDIDATES.copy()
    historic['historical_score_on_new_grid']=(.6*minmax(historic.r)
                                              +.1*minmax(-historic.rmse_z)
                                              +.3*minmax(historic.n_pct))
    historic=historic.sort_values('historical_score_on_new_grid',ascending=False)
    historic.to_csv(OUT/'historical_weights_on_new_grid_NOT_Table4.csv',index=False)
    print('Original min-max 0.6/0.1/0.3 form on the new candidate grid (not Table 4):',
          historic.iloc[0][['scale','tau','window_days','r','n_pct']].to_dict())
    print('Maximum paired-z identity discrepancy:',
          float(np.nanmax(abs(LOCAL_CANDIDATES.rmse_z**2-2*(1-LOCAL_CANDIDATES.r)))))

    plateau_rows=[]
    for delta in [.01,.02,.05]:
        q=plateau_candidates(MATRIX,delta)
        q=q[q.scale.isin(['G1','G2','G3'])]
        for lam in [.3,.5,.7]:
            if len(q):
                x=primary_score(q,lam).iloc[0]
                plateau_rows.append({'delta_r':delta,'lambda_alignment':lam,'scale':x.scale,
                                     'tau':x.tau,'window_days':x.window_days,'r':x.r,'n_pct':x.n_pct})
    PLATEAU_CHECK=pd.DataFrame(plateau_rows)
    PLATEAU_CHECK.to_csv(OUT/'window_plateau_tolerance_sensitivity.csv',index=False)
    # Explicit 231-point simplex sweep of the manuscript's three score weights.
    # All axes use fixed theoretical scales; candidate-set min/max cannot move the goalposts.
    score_candidates=LOCAL_CANDIDATES.copy()
    score_candidates['setting']=(score_candidates.scale+' / τ='+score_candidates.tau.astype(int).astype(str)
                                 +'% / '+score_candidates.window_days.astype(int).astype(str)+' d')
    score_candidates['r01']=(score_candidates.r+1)/2
    score_candidates['error01']=1-score_candidates.rmse_z/2
    score_candidates['n01']=score_candidates.n_pct/100
    simplex_rows=[]
    top5_count={s:0 for s in score_candidates.setting}
    winner_count={s:0 for s in score_candidates.setting}
    for ar in range(21):
        for ae in range(21-ar):
            an=20-ar-ae
            wr,we,wn=ar/20,ae/20,an/20
            q=score_candidates.copy()
            q['score3']=wr*q.r01+we*q.error01+wn*q.n01
            q=q.sort_values(['score3','n_pct','window_days','setting'],
                            ascending=[False,False,True,True])
            winner=q.iloc[0]
            winner_count[winner.setting]+=1
            for setting in q.head(5).setting:
                top5_count[setting]+=1
            simplex_rows.append({'w_r':wr,'w_error':we,'w_n':wn,
                                 'selected_setting':winner.setting,'selected_scale':winner.scale,
                                 'selected_tau':int(winner.tau),'selected_window_days':int(winner.window_days),
                                 'selected_r':winner.r,'selected_rmse_z':winner.rmse_z,
                                 'selected_n_pct':winner.n_pct,'score':winner.score3})
    WEIGHT_SIMPLEX=pd.DataFrame(simplex_rows)
    assert len(WEIGHT_SIMPLEX)==231
    WEIGHT_SIMPLEX.to_csv(OUT/'composite_score_231_weight_simplex.csv',index=False)
    WEIGHT_FREQUENCY=score_candidates[['setting','scale','tau','window_days','r','rmse_z','n_pct']].copy()
    WEIGHT_FREQUENCY['winner_share']=WEIGHT_FREQUENCY.setting.map(winner_count)/len(WEIGHT_SIMPLEX)
    WEIGHT_FREQUENCY['top_five_share']=WEIGHT_FREQUENCY.setting.map(top5_count)/len(WEIGHT_SIMPLEX)
    WEIGHT_FREQUENCY=WEIGHT_FREQUENCY.sort_values(['top_five_share','winner_share'],ascending=False)
    WEIGHT_FREQUENCY.to_csv(OUT/'composite_score_weight_stability.csv',index=False)
    HISTORICAL_SELECTION=WEIGHT_SIMPLEX[(WEIGHT_SIMPLEX.w_r==.6)&
                                        (WEIGHT_SIMPLEX.w_error==.1)&
                                        (WEIGHT_SIMPLEX.w_n==.3)].iloc[0]
    print('Fixed-scale 231-weight audit; 0.6/0.1/0.3 selects:',HISTORICAL_SELECTION.selected_setting)

### Figure 5 — The candidate frontier and score-weight sensitivity

The frontier shows $r$ against direct **N%** for each shortest-near-plateau setting. Figure 5B shows how the two-axis winner changes over the full $\lambda$ range. Figure 5C maps all **231 historical-style weight triples** on a simplex, with 0.6/0.1/0.3 highlighted. Figure 5D reports each setting's share of wins and top-five placements. A broad stable region supports a robust *selection rule*; rapid switching means the manuscript should report a family of application-dependent choices rather than one supposedly universal weight vector. These charts select against local NGCP; Section 5 checks held-out transfer.


In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    fig=go.Figure()
    for scale in [v[0] for v in GROUPS.values()]:
        q=LOCAL_CANDIDATES[LOCAL_CANDIDATES.scale==scale]
        if q.empty: continue
        fig.add_trace(go.Scatter(x=q.n_pct,y=q.r,mode='markers',name=scale,
                                 marker=dict(size=9,color=COLOR[scale],opacity=.72),
                                 customdata=np.stack([q.tau,q.window_days],axis=1),
                                 hovertemplate=f'{scale}<br>SC τ=%{{customdata[0]}}%<br>Window=%{{customdata[1]}} d<br>Direct N=%{{x:.1f}}%<br>r=%{{y:.3f}}<extra></extra>'))
    fig.update_layout(template='plotly_white',width=1030,height=570,
                      title='Candidate frontier after the shortest-near-plateau window rule',
                      xaxis_title='Direct N% of load dates',yaxis_title='Pearson r',
                      legend=dict(orientation='h',y=-.21),margin=dict(l=70,r=25,t=75,b=115),font=dict(color='#233B60'))
    export_figure((fig).update_layout(template='plotly_white', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='#233B60', size=13)), FIG_DIAGNOSTIC / ('fig_05a_alignment_coverage_candidate_frontier'), scale=2)

    fig=make_subplots(rows=2,cols=1,shared_xaxes=True,vertical_spacing=.10,
                      subplot_titles=['Selected SC threshold and mask','Alignment and direct-date coverage of selection'])
    for scale in [v[0] for v in GROUPS.values()]:
        q=LOCAL_WEIGHTS[LOCAL_WEIGHTS.selected_scale==scale]
        if q.empty: continue
        fig.add_trace(go.Scatter(x=q.lambda_alignment,y=q.selected_tau,mode='markers',name=scale,
                                 marker=dict(size=10,color=COLOR[scale]),
                                 text=[f'{w} d' for w in q.selected_window_days],
                                 hovertemplate='Alignment weight %{x:.2f}<br>SC τ=%{y}%<br>Window %{text}<extra>'+scale+'</extra>'),row=1,col=1)
    fig.add_trace(go.Scatter(x=LOCAL_WEIGHTS.lambda_alignment,y=LOCAL_WEIGHTS.r,mode='lines+markers',
                             name='Selected r',line=dict(color='#4E79A7',width=2)),row=2,col=1)
    fig.add_trace(go.Scatter(x=LOCAL_WEIGHTS.lambda_alignment,y=LOCAL_WEIGHTS.n_pct/100,
                             mode='lines+markers',name='Selected direct fraction',
                             line=dict(color='#375623',width=2)),row=2,col=1)
    fig.update_xaxes(title_text='Declared alignment priority λ',row=2,col=1)
    fig.update_yaxes(title_text='SC threshold τ (%)',row=1,col=1)
    fig.update_yaxes(title_text='r / fraction',row=2,col=1)
    fig.update_layout(template='plotly_white',width=1110,height=660,
                      title='What changes when the application values alignment more than temporal coverage?',
                      legend=dict(orientation='h',y=-.13),margin=dict(l=75,r=25,t=80,b=110),font=dict(color='#233B60'))
    export_figure((fig).update_layout(template='plotly_white', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='#233B60', size=13)), FIG_DIAGNOSTIC / ('fig_05b_local_weight_sensitivity'), scale=2)
    # 231-combination historical-style weight sensitivity, using the same palette as the RQ1 masks.
    fig=go.Figure()
    for scale in [v[0] for v in GROUPS.values()]:
        q=WEIGHT_SIMPLEX[WEIGHT_SIMPLEX.selected_scale==scale]
        if q.empty: continue
        fig.add_trace(go.Scatterternary(a=q.w_r,b=q.w_error,c=q.w_n,mode='markers',name=scale,
                                        marker=dict(size=8,color=COLOR[scale],opacity=.8),
                                        text=q.selected_setting,
                                        hovertemplate='r weight %{a:.2f}<br>Error weight %{b:.2f}<br>N% weight %{c:.2f}<br>Winner %{text}<extra></extra>'))
    fig.add_trace(go.Scatterternary(a=[.6],b=[.1],c=[.3],mode='markers',name='Manuscript 0.6/0.1/0.3',
                                    marker=dict(symbol='star',size=17,color='#D65A4A',line=dict(color='#233B60',width=1)),
                                    text=[HISTORICAL_SELECTION.selected_setting],
                                    hovertemplate='Manuscript weights<br>Winner %{text}<extra></extra>'))
    fig.update_layout(template='plotly_white',width=930,height=740,
                      title='Composite score: winning GHSL setting across 231 weight triples',
                      ternary=dict(sum=1,aaxis=dict(title='r weight'),
                                   baxis=dict(title='inverse-RMSE weight'),
                                   caxis=dict(title='N% weight')),
                      legend=dict(orientation='h',y=-.06),margin=dict(l=55,r=55,t=80,b=115),
                      font=dict(color='#233B60',size=13))
    export_figure((fig).update_layout(template='plotly_white', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='#233B60', size=13)), FIG_DIAGNOSTIC / ('fig_05c_composite_score_231_weight_simplex'), scale=2)

    q=WEIGHT_FREQUENCY.head(12).sort_values('top_five_share')
    fig=go.Figure()
    fig.add_trace(go.Bar(x=q.top_five_share,y=q.setting,orientation='h',name='Top-five share',
                         marker_color='#4E79A7'))
    fig.add_trace(go.Scatter(x=q.winner_share,y=q.setting,mode='markers',name='Winner share',
                             marker=dict(color='#D65A4A',size=10)))
    fig.update_layout(template='plotly_white',width=1190,height=660,
                      title='Configuration stability across 231 composite-score weights',
                      xaxis_title='Share of weight combinations',xaxis_range=[0,1],
                      legend=dict(orientation='h',y=-.12),margin=dict(l=265,r=30,t=80,b=100),
                      font=dict(color='#233B60',size=13))
    export_figure((fig).update_layout(template='plotly_white', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='#233B60', size=13)), FIG_DIAGNOSTIC / ('fig_05d_composite_score_weight_stability'), scale=2)

## 5. Does the selection rule transfer when NGCP is unavailable locally?

A score containing $r$ or RMSE **cannot be recomputed in a target region without NGCP**. The testable alternative is to learn a fixed mask × SC × window policy on other island grids, apply it unchanged to a held-out grid, then reveal that grid's NGCP only for evaluation. This uses the RQ1 Notebook 04 CSVs and workbook; the loader audits the actual Hour 1 column selected on each sheet, including Negros. The intended design is five leave-one-grid-out folds. If a sheet cannot be safely parsed, the notebook reports it and runs the remaining eligible folds, without silently substituting a different load hour.

For each held-out grid and declared two-axis weight, the code chooses a shortest near-plateau window and ranks common candidates using training grids only. It compares held-out $r$ and N% against G8 τ=0, one day, and a clearly labelled *hindsight* local selection. The G8 comparator is GHSL classes 10–30, not the unmasked study area. Its values have the same P95 upper cap. A transfer result supports the **selection procedure**, subject to the number of valid folds; it cannot make load-based score components available where NGCP is absent.


In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    SHEET_MAP={
        'Panay':'PANAY HOURLY LOAD 2013-2024',
        'Negros':'NEGROS HOURLY LOAD 2013-2024',
        'Samar-Leyte':'LEY-SAM HOURLY LOAD 2013-2024',
        'Cebu':'CEBU HOURLY LOAD 2013-2024',
        'Bohol':'BOHOL LOAD 2013-2024',
    }
    REGION_NAMES={'SamarLeyte':'Samar-Leyte','BoholProvince':'Bohol',
                  'CebuProvince':'Cebu','NegrosIsland':'Negros','PanayIsland':'Panay'}

    def is_hour_one(label):
        s=re.sub(r'\s+','',str(label).strip().lower()).replace('a.m.','am')
        return bool(re.fullmatch(r'(?:hour|h)?0?1(?:\.0)?|0?1(?::00){1,2}(?:am)?|0?1am',s))

    def read_ngcp_hour_one(workbook,sheet):
        diagnostics=[]
        for header in (1,0,2,3):
            f=pd.read_excel(workbook,sheet_name=sheet,header=header)
            f.columns=[str(x).strip() for x in f.columns]
            dates=pd.to_datetime(f.iloc[:,0],errors='coerce').dt.normalize()
            if dates.notna().sum()<30:
                diagnostics.append((header,list(f.columns[:8]),'fewer than 30 dates'))
                continue
            hour_col=next((c for c in f.columns[1:] if is_hour_one(c)),None)
            if hour_col is not None:
                values=pd.to_numeric(f[hour_col],errors='coerce')
                if values.notna().sum()>=30:
                    return f,dates,hour_col,header,'explicit Hour 1 label'
            diagnostics.append((header,list(f.columns[:8]),'no explicit Hour 1 label'))
        # RQ1 Notebook 04 uses this position fallback. Keep it visible and audited.
        f=pd.read_excel(workbook,sheet_name=sheet,header=1)
        f.columns=[str(x).strip() for x in f.columns]
        dates=pd.to_datetime(f.iloc[:,0],errors='coerce').dt.normalize()
        numeric_like=[c for c in f.columns[1:] if re.fullmatch(r'\d+(?:\.0)?',str(c))]
        candidates=numeric_like or list(f.columns[1:])
        if dates.notna().sum()>=30 and candidates:
            hour_col=candidates[0]
            if pd.to_numeric(f[hour_col],errors='coerce').notna().sum()>=30:
                return f,dates,hour_col,1,'Notebook 04 positional fallback — verify header'
        raise ValueError(f'{sheet}: Hour 1 could not be identified. Header diagnostics: {diagnostics}')

    def cross_region_inputs(folder,workbook):
        if not folder.is_dir() or not workbook.is_file():
            return {},{},f'Missing RQ1 Notebook 04 data: {folder.relative_to(PROJECT_ROOT)} or {workbook.relative_to(PROJECT_ROOT)}'
        files=sorted(folder.glob('*_DNBBRDF_class*_stats.csv'))
        if not files:
            return {},{},f'No Notebook 04 mask CSVs found in {folder.relative_to(PROJECT_ROOT)}'
        region_data={}
        for path in files:
            parts=path.name.replace('_stats.csv','').split('_DNBBRDF_class')
            if len(parts)!=2 or parts[0] not in REGION_NAMES: continue
            region=REGION_NAMES[parts[0]]
            group='30' if parts[1]=='30-only' else parts[1]
            if group not in GROUPS: continue
            region_data[(region,GROUPS[group][0])]=prep_direct(read_daily_csv(path,'Mean_by_sum'))[0]
        loads={}; audit=[]; skipped=[]
        for region,sheet in SHEET_MAP.items():
            try:
                f,dates,hour_col,header,method=read_ngcp_hour_one(workbook,sheet)
                series=pd.Series(pd.to_numeric(f[hour_col],errors='coerce').to_numpy(),index=dates)
                series=series[series.index.notna()].dropna()
                if series.index.duplicated().any(): raise ValueError('duplicate dates')
                loads[region]=series.reindex(CALENDAR)
                audit.append({'region':region,'sheet':sheet,'hour_column':str(hour_col),
                              'excel_header_row':header+1,'selection_method':method,
                              'n_load_dates':int(loads[region].notna().sum())})
                print(f'{region}: Hour 1 = {hour_col!r}, Excel header row {header+1} ({method})')
            except (ValueError,KeyError) as exc:
                skipped.append(f'{region}: {type(exc).__name__}')
        pd.DataFrame(audit).to_csv(OUT/'cross_island_ngcp_hour1_column_audit.csv',index=False)
        for message in skipped: print('NGCP parsing:',message)
        complete=[region for region in loads if all((region,g) in region_data for g in [v[0] for v in GROUPS.values()])]
        if len(complete)<3:
            return {},{},f'Only {len(complete)} island grids have both Hour 1 load and all GHSL masks; at least three are needed.'
        return ({key:value for key,value in region_data.items() if key[0] in complete},
                {region:loads[region] for region in complete},'')

    CROSS_DATA,CROSS_LOAD,CROSS_STATUS=cross_region_inputs(CROSS_DIR,NGCP_XLSX)
    print(CROSS_STATUS or f'Loaded {len(CROSS_DATA)} region × mask direct inputs and {len(CROSS_LOAD)} NGCP grids.')

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    CROSS_MATRIX=pd.DataFrame()
    if not CROSS_STATUS:
        cross_rows=[]
        for (region,scale),d in CROSS_DATA.items():
            load=CROSS_LOAD[region]
            for tau in TRANSFER_TAUS:
                for w in WINDOWS:
                    cross_rows.append({'region':region,'scale':scale,'tau':tau,'window_days':w,
                                       'lens':'direct',**configuration_metric(d,load,tau,w,'direct')})
        CROSS_MATRIX=pd.DataFrame(cross_rows)
        CROSS_MATRIX.to_csv(OUT/'cross_island_mask_sc_window_metric_cube.csv',index=False)
        print(f'Saved {len(CROSS_MATRIX):,} direct-date cross-island candidates.')
    else:
        (OUT/'cross_island_status.txt').write_text(CROSS_STATUS+'\n')

    # Optional supplementary comparison using only the direct summaries available for all islands.
    # A 4-day rolling mean of regional CSV values is NOT the pixel-first Roman-inspired block composite.
    if not CROSS_MATRIX.empty:
        comparison=CROSS_MATRIX[(CROSS_MATRIX.tau==0)&(CROSS_MATRIX.window_days.isin([1,4]))].copy()
        comparison['method']=np.where(comparison.window_days==1,'Direct daily regional summary',
                                      '4-day calendar mean of regional summaries')
        comparison[['region','scale','method','r','rmse_z','n_pairs','n_pct']].to_csv(
            OUT/'supplement_cross_island_direct_summary_comparison_NO_gapfilled.csv',index=False)

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    def transfer_fold(matrix,holdout,lam,delta,region_data=None,region_load=None):
        train=matrix[matrix.region!=holdout].copy()
        train=train[train.r.notna()&(train.n_pairs>=30)]
        target=matrix[matrix.region==holdout].copy()
        regions=set(train.region.unique())
        n_training=len(set(matrix.region.unique()))-1
        if len(regions)!=n_training or n_training<2:
            raise ValueError(f'{holdout}: expected {n_training} training island grids')
        # A candidate must be present and sufficiently observed in every training region.
        agg=(train.groupby(['scale','tau','window_days'],as_index=False)
             .agg(median_r=('r','median'),median_pct=('n_pct','median'),
                  min_pct=('n_pct','min'),regions=('region','nunique')))
        agg=agg[(agg.regions==n_training)&(agg.min_pct>=100*MIN_DIRECT_COVERAGE)]
        choices=[]
        for _,g in agg.groupby(['scale','tau'],sort=False):
            best=g.median_r.max()
            choices.append(g[g.median_r>=best-delta].sort_values('window_days').iloc[0])
        if not choices:
            raise ValueError(f'{holdout}: no transfer candidate survives training coverage rule')
        choices=pd.DataFrame(choices)
        choices['training_score']=lam*(choices.median_r+1)/2+(1-lam)*choices.median_pct/100
        selected=choices.sort_values(['training_score','median_pct','window_days','scale','tau'],
                                     ascending=[False,False,True,True,True]).iloc[0]
        key=(target.scale==selected.scale)&(target.tau==selected.tau)&(target.window_days==selected.window_days)
        if key.sum()!=1:
            raise ValueError(f'{holdout}: selected configuration missing from held-out metric matrix')
        evaluated=target[key].iloc[0]
        baseline=target[(target.scale=='G8')&(target.tau==0)&(target.window_days==1)].iloc[0]
        valid_heldout=bool(np.isfinite(evaluated.r) and evaluated.n_pairs>=30)
        # Hindsight comparison uses held-out load and the SAME near-plateau rule.
        # It is reported only as an optimistic benchmark, never used for selection.
        hindsight=target[target.r.notna()&(target.n_pairs>=30)&
                         (target.n_pct>=100*MIN_DIRECT_COVERAGE)].copy()
        local_choices=[]
        for _,g in hindsight.groupby(['scale','tau'],sort=False):
            best=g.r.max()
            local_choices.append(g[g.r>=best-delta].sort_values('window_days').iloc[0])
        if not local_choices:
            raise ValueError(f'{holdout}: no eligible hindsight comparison')
        local_choices=pd.DataFrame(local_choices)
        local_choices['oracle_score']=lam*(local_choices.r+1)/2+(1-lam)*local_choices.n_pct/100
        oracle=local_choices.sort_values(['oracle_score','n_pct'],ascending=False).iloc[0]

        # Isolate spatial/SC choices: compare selected NTL with G8 at the SAME
        # window, against the SAME smoothed load, and on the SAME calendar dates.
        selected_common_r, g8_common_r, common_n=np.nan,np.nan,0
        if region_data is not None and region_load is not None:
            held_load=region_load[holdout]
            sx,sy,_,_=smooth_pair(region_data[(holdout,selected.scale)],held_load,
                                  int(selected.tau),int(selected.window_days),'direct')
            bx,_,_,_=smooth_pair(region_data[(holdout,'G8')],held_load,0,
                                 int(selected.window_days),'direct')
            common=held_load.index[sx.notna()&bx.notna()&sy.notna()]
            sm=paired_metrics(sx,sy,common)
            bm=paired_metrics(bx,sy,common)
            selected_common_r,g8_common_r,common_n=sm['r'],bm['r'],sm['n_pairs']
        return {'held_out':holdout,'lambda_alignment':lam,'delta_r':delta,
                'selected_mask':selected.scale,'selected_tau':int(selected.tau),
                'selected_window_days':int(selected.window_days),
                'training_score':float(selected.training_score),
                'held_out_r':evaluated.r,'held_out_rmse_z':evaluated.rmse_z,
                'held_out_n':int(evaluated.n_pairs),'held_out_n_pct':evaluated.n_pct,
                'valid_heldout':valid_heldout,
                'selected_common_r':selected_common_r,'g8_same_window_common_r':g8_common_r,
                'common_pair_n':common_n,
                'baseline_G8_raw_r':baseline.r,'baseline_G8_raw_n_pct':baseline.n_pct,
                'hindsight_oracle_r':oracle.r,'hindsight_oracle_n_pct':oracle.n_pct,
                'hindsight_oracle_mask':oracle.scale,'hindsight_oracle_tau':int(oracle.tau),
                'hindsight_oracle_window_days':int(oracle.window_days)}

    TRANSFER=pd.DataFrame()
    if not CROSS_MATRIX.empty:
        transfer_rows=[]
        for holdout in CROSS_LOAD:
            for lam in [.3,.5,.7]:
                for delta in [.01,.02,.05]:
                    try:
                        transfer_rows.append(transfer_fold(CROSS_MATRIX,holdout,lam,delta,CROSS_DATA,CROSS_LOAD))
                    except ValueError as exc:
                        print('Transfer fold skipped:',exc)
        TRANSFER=pd.DataFrame(transfer_rows)
        TRANSFER.to_csv(OUT/'leave_one_island_out_weight_and_window_transfer.csv',index=False)
        print(f'Saved {len(TRANSFER)} held-out evaluations across {len(CROSS_LOAD)} eligible grids.')
        if not TRANSFER.empty and (~TRANSFER.valid_heldout).any():
            print('ATTENTION: Some selected held-out configurations have fewer than 30 pairs or undefined r.')
            display(TRANSFER.loc[~TRANSFER.valid_heldout,
                             ['held_out','lambda_alignment','delta_r','selected_mask','selected_tau',
                              'selected_window_days','held_out_n']])

### Figure 6 — Generalizability is a held-out question

This chart displays the predeclared `λ=0.5, Δr=0.02` policy for each held-out island. It shows held-out $r$ **and** direct-date fraction, alongside the capped G8 τ=0, one-day comparator and the held-out *hindsight* best score (using the same window rule). That whole-pipeline G8 comparison changes smoothing as well as mask. The exported table adds a **same-window, same-date G8 comparison** to isolate the mask/SC contribution. Figure 6B then checks the held-out average across all declared weights and window tolerances; cells are hidden unless all available folds have valid $r$. If the transferred choices remain useful across grids, that supports transport of the **selection logic**; if they vary or underperform, report that limitation and consider a smaller family of application-specific policies. It does **not** justify calculating an $r$-weighted score where load is unavailable.

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    if not TRANSFER.empty:
        q=TRANSFER[(TRANSFER.lambda_alignment==.5)&(TRANSFER.delta_r==.02)].copy()
        q=q[q.valid_heldout].copy()
        if q.empty:
            raise ValueError('No valid held-out fold for the declared λ=0.5, Δr=0.02 display')
        order=list(CROSS_LOAD)
        q['held_out']=pd.Categorical(q.held_out,categories=order,ordered=True)
        q=q.sort_values('held_out')
        fig=make_subplots(rows=1,cols=2,subplot_titles=['Held-out Pearson r','Held-out directly paired load dates (%)'],
                          horizontal_spacing=.11)
        for col,triples in [(1,[('G8 τ=0, 1 d','baseline_G8_raw_r','#A8CE74'),
                               ('Transferred rule','held_out_r','#4E79A7'),
                               ('Hindsight local best','hindsight_oracle_r','#D65A4A')]),
                            (2,[('G8 τ=0, 1 d','baseline_G8_raw_n_pct','#A8CE74'),
                                ('Transferred rule','held_out_n_pct','#4E79A7'),
                                ('Hindsight local best','hindsight_oracle_n_pct','#D65A4A')])]:
            for name,key,color in triples:
                fig.add_trace(go.Scatter(x=q.held_out.astype(str),y=q[key],mode='markers+lines',
                                         name=name,showlegend=(col==1),
                                         marker=dict(size=10,color=color),line=dict(color=color,width=1.5)),row=1,col=col)
        fig.update_yaxes(title_text='Pearson r',row=1,col=1)
        fig.update_yaxes(title_text='Directly paired dates (%)',row=1,col=2)
        fig.update_xaxes(title_text='Held-out island grid',tickangle=-20)
        fig.update_layout(template='plotly_white',width=1250,height=510,
                          title='Cross-island transfer of a fixed reliability policy (λ=0.5; Δr=0.02)',
                          legend=dict(orientation='h',y=-.22),margin=dict(l=70,r=30,t=80,b=120),font=dict(color='#233B60'))
        export_figure((fig).update_layout(template='plotly_white', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='#233B60', size=13)), FIG_DIAGNOSTIC / ('fig_06_held_out_rule_transfer'), scale=2)
        display(q[['held_out','selected_mask','selected_tau','selected_window_days',
                   'held_out_r','held_out_n','held_out_n_pct','common_pair_n',
                   'selected_common_r','g8_same_window_common_r']].round(3))

        summary=(TRANSFER[TRANSFER.valid_heldout]
                 .groupby(['lambda_alignment','delta_r'],as_index=False)
                 .agg(mean_heldout_r=('held_out_r','mean'),
                      mean_heldout_n_pct=('held_out_n_pct','mean'),
                      valid_folds=('held_out','nunique')))
        summary.loc[summary.valid_folds<len(CROSS_LOAD),['mean_heldout_r','mean_heldout_n_pct']]=np.nan
        summary.to_csv(OUT/'transfer_priority_sensitivity_summary.csv',index=False)
        fig=make_subplots(rows=1,cols=2,subplot_titles=['Mean held-out r','Mean direct-date retention (%)'],
                          horizontal_spacing=.15)
        for col,field,zmin,zmax,title in [(1,'mean_heldout_r',-1,1,'Pearson r'),
                                         (2,'mean_heldout_n_pct',0,100,'N%')]:
            p=summary.pivot(index='delta_r',columns='lambda_alignment',values=field)
            fig.add_trace(go.Heatmap(x=p.columns,y=p.index,z=p.values,colorscale='Blues',
                                     zmin=zmin,zmax=zmax,colorbar=dict(title=title),
                                     hovertemplate='λ=%{x}<br>Δr=%{y}<br>Mean=%{z:.3f}<extra></extra>'),row=1,col=col)
        fig.update_xaxes(title_text='Alignment priority λ')
        fig.update_yaxes(title_text='Near-plateau tolerance Δr',row=1,col=1)
        fig.update_layout(template='plotly_white',width=1150,height=470,
                          title=f'Held-out sensitivity across priorities (complete {len(CROSS_LOAD)}-fold cells only)',
                          margin=dict(l=80,r=80,t=80,b=70),font=dict(color='#233B60'))
        export_figure((fig).update_layout(template='plotly_white', paper_bgcolor='rgba(0,0,0,0)', font=dict(color='#233B60', size=13)), FIG_DIAGNOSTIC / ('fig_06b_held_out_priority_sensitivity'), scale=2)
    else:
        print('Figure 6 awaits the RQ1 Notebook 04 cross-island input folder and NGCP workbook; no transfer claim is made.')

## 6. Additional supplementary checks from Notebook 06

Use the Samar–Leyte source exports to reproduce the earlier moving-average audit, SC-qualified mask time series, support-rule sensitivity and year-by-year checks. The separate source calendar and validation period in these diagnostics are defined in the original setup. These diagnostics use the 2013–2024 validation period and retain the original support-rule definitions.

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    OUTPUT_ROOT = ROOT / "outputs"
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    def daily_pair_metrics(signal, admitted, load):
        pair = pd.DataFrame({
            "ntl": signal.where(admitted),
            "load": load,
        }).dropna()

        n_pct = 100 * len(pair) / load.notna().sum()
        r = rmse_z = np.nan

        if (len(pair) >= 3
                and pair["ntl"].std(ddof=0) > 0
                and pair["load"].std(ddof=0) > 0):
            x = (pair["ntl"] - pair["ntl"].mean()) / pair["ntl"].std(ddof=0)
            y = (pair["load"] - pair["load"].mean()) / pair["load"].std(ddof=0)
            r = float(x.corr(y))
            rmse_z = float(np.sqrt(np.mean((x - y) ** 2)))

        return len(pair), n_pct, r, rmse_z

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    # Shared RQ1 regional inputs for the thematic-mask figures below.







    MASK_DIR = ROOT / "datasets/vnp46a2/ghsl_masked_ntl_samar_leyte"
    NGCP_CSV = ROOT / "datasets/ngcp/NGCP_Hourly_Demand.csv"
    OUT_DIR = FIG_DIAGNOSTIC
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    DATES = pd.date_range("2013-01-01", "2024-12-31", freq="D")

    GROUPS = [
        ("G8", "10-30", "All Land", "#4E79A7"),
        ("G7", "11-30", "Rural Settlements", "#A8CE74"),
        ("G6", "12-30", "Low–Mid Rural", "#7BAF54"),
        ("G5", "13-30", "Built-Up / Rural Mix", "#375623"),
        ("G4", "21-30", "Urban Fringe", "#D9B900"),
        ("G3", "22-30", "Core & Inner", "#E6A23C"),
        ("G2", "23-30", "Dense Urban", "#A87000"),
        ("G1", "30", "Urban Core", "#D65A4A"),
    ]

    def read_ngcp_1am():
        if not NGCP_CSV.is_file():
            raise FileNotFoundError(NGCP_CSV.relative_to(PROJECT_ROOT))
        f = pd.read_csv(NGCP_CSV, skiprows=2)
        if f.shape[1] < 4:
            raise ValueError("NGCP CSV layout differs from the RQ1 hourly file")
        f.columns = (["Date", "Coincident Peak", "Hour No"]
                     + [str(i) for i in range(1, f.shape[1] - 2)])
        f["Date"] = pd.to_datetime(f["Date"], dayfirst=True, errors="coerce").dt.normalize()
        f["1"] = pd.to_numeric(f["1"], errors="coerce")
        f = f.dropna(subset=["Date"]).set_index("Date").sort_index()
        if f.index.duplicated().any():
            raise ValueError("Duplicate NGCP dates")
        return f["1"].reindex(DATES)

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    # None displays raw radiance. Set to 95 later to cap values at P95.
    CAP_PERCENTILE = 95
    MARK_ABOVE_PERCENTILE = 95
    SC_THRESHOLDS = [None, 25, 50, 75]

    def read_direct_mask(suffix):
        path = MASK_DIR / f"EasternVisayas_DNBBRDF_{suffix}_stats.csv"
        if not path.is_file():
            raise FileNotFoundError(path.relative_to(PROJECT_ROOT))

        d = pd.read_csv(path)
        if "date" not in d:
            if "system:index" not in d:
                raise ValueError(f"{path.name}: no date or system:index")
            d["date"] = (
                d["system:index"].astype(str)
                .str.extract(r"(\d{4}_\d{2}_\d{2})")[0]
                .str.replace("_", "-", regex=False)
            )

        d["date"] = pd.to_datetime(d["date"], errors="raise").dt.normalize()
        if d["date"].duplicated().any():
            raise ValueError(f"{path.name}: duplicate dates")

        d = d.set_index("date").sort_index()
        d["Valid_px"] = pd.to_numeric(d["Valid_px"], errors="coerce")
        d["raw"] = pd.to_numeric(d["Mean_by_sum"], errors="coerce")

        # Use the full export to define each thematic mask's SC denominator.
        denominator = d["Valid_px"].max()
        if not np.isfinite(denominator) or denominator <= 0:
            raise ValueError(f"{path.name}: no valid pixels")

        d["SC"] = 100 * d["Valid_px"] / denominator
        d["raw"] = d["raw"].where(d["Valid_px"] > 0)
        return d[["raw", "SC"]].reindex(DATES)


    # Load each mask once; all four figures use identical source values.
    MASK_DATA = {
        code: read_direct_mask(suffix)
        for code, suffix, label, color in GROUPS
    }
    LOAD = read_ngcp_1am()
    ORDERED_GROUPS = list(reversed(GROUPS))  # G1 through G8

    def plot_daily_masks(sc_threshold, display_figure=True):
        threshold_label = (
            "No SC threshold" if sc_threshold is None
            else f"SC ≥ {sc_threshold}%"
        )

        fig = make_subplots(
            rows=9, cols=1,
            shared_xaxes=True,
            vertical_spacing=0.012,
            row_heights=[0.18] + [0.1025] * 8,
        )

        fig.add_trace(
            go.Scattergl(
                x=DATES, y=LOAD,
                mode="lines",
                line=dict(color="#171717", width=1.7),
                name="NGCP load (1 AM)",
                connectgaps=False,
                hovertemplate=(
                    "%{x|%d %b %Y}<br>%{y:.1f} MW"
                    "<extra>NGCP load</extra>"
                ),
            ),
            row=1, col=1,
        )
        fig.update_yaxes(
            title_text="MW", rangemode="tozero",
            nticks=3, gridcolor="#E8EDF3",
            row=1, col=1,
        )
        fig.add_annotation(
            x=1.01, xref="paper", y=0.5, yref="y domain",
            text="NGCP load · 1 AM",
            showarrow=False, xanchor="left",
            font=dict(size=11, color="#233B60"),
        )

        coverage = []

        for row, (code, suffix, label, color) in enumerate(
            ORDERED_GROUPS, start=2
        ):
            d = MASK_DATA[code]
            raw = d["raw"]
            sc = d["SC"]

            admitted = raw.notna()
            if sc_threshold is not None:
                admitted &= sc >= sc_threshold

            # One fixed, unfiltered reference distribution per mask.
            # Its percentile does not change between the four figures.
            mark_limit = raw.quantile(MARK_ABOVE_PERCENTILE / 100)

            if CAP_PERCENTILE is None:
                shown = raw.copy()
                high = admitted & (raw > mark_limit)
            else:
                cap_limit = raw.quantile(CAP_PERCENTILE / 100)
                shown = raw.clip(upper=cap_limit)
                high = admitted & (raw > cap_limit)

            ordinary = admitted & ~high
            custom = np.column_stack([
                sc.to_numpy(dtype=float),
                raw.to_numpy(dtype=float),
            ])

            # Small dots are the actual daily direct observations.
            fig.add_trace(
                go.Scattergl(
                    x=DATES,
                    y=shown.where(ordinary),
                    mode="markers",
                    marker=dict(size=3, color=color, opacity=0.52),
                    name=f"{code}: {label}",
                    customdata=custom,
                    hovertemplate=(
                        "%{x|%d %b %Y}<br>"
                        "Raw %{customdata[1]:.2f} nW·cm⁻²·sr⁻¹<br>"
                        "SC %{customdata[0]:.1f}%"
                        f"<extra>{code}: {label}</extra>"
                    ),
                ),
                row=row, col=1,
            )

            # Open triangles identify values above the selected percentile.
            # With CAP_PERCENTILE=None, they remain at their RAW values.
            fig.add_trace(
                go.Scatter(
                    x=DATES,
                    y=shown.where(high),
                    mode="markers",
                    marker=dict(
                        symbol="triangle-up-open",
                        size=6,
                        color=color,
                        line=dict(color="#233B60", width=0.8),
                    ),
                    name=f"{code} upper tail",
                    customdata=custom,
                    hovertemplate=(
                        "%{x|%d %b %Y}<br>"
                        "Raw %{customdata[1]:.2f} nW·cm⁻²·sr⁻¹<br>"
                        "Displayed %{y:.2f}<br>"
                        "SC %{customdata[0]:.1f}%"
                        f"<extra>{code}: {label} · upper tail</extra>"
                    ),
                ),
                row=row, col=1,
            )

            n_pairs, n_pct, r, rmse_z = daily_pair_metrics(
                shown, admitted, LOAD
            )

            coverage.append({
                "mask": code,
                "SC threshold": threshold_label,
                "paired days": n_pairs,
                "N%": n_pct,
                "r": r,
                "RMSE_z": rmse_z,
            })

            fig.update_yaxes(
                rangemode="tozero", nticks=3,
                gridcolor="#E8EDF3",
                tickfont=dict(size=10),
                row=row, col=1,
            )

            r_text = "—" if not np.isfinite(r) else f"{r:.2f}"
            rmse_text = "—" if not np.isfinite(rmse_z) else f"{rmse_z:.2f}"

            fig.add_annotation(
                x=1.01, xref="paper",
                y=0.5, yref=f"y{row} domain",
                text=(
                    f"{code} · {label}<br>"
                    f"r = {r_text}<br>"
                    f"RMSE (z) = {rmse_text}<br>"
                    f"N% = {n_pct:.1f}%"
                ),
                showarrow=False,
                xanchor="left",
                align="left",
                font=dict(size=10, color="#233B60"),
            )

        for day, label in [
            ("2013-11-08", "Haiyan"),
            ("2021-12-16", "Rai"),
        ]:
            event = pd.Timestamp(day)
            for row in range(1, 10):
                fig.add_vline(
                    x=event, row=row, col=1,
                    line_dash="dash",
                    line_color="#233B60",
                    line_width=1,
                )
            fig.add_annotation(
                x=event, y=1.005, yref="paper",
                text=label, showarrow=False,
                yanchor="bottom", bgcolor="white",
                font=dict(size=11, color="#233B60"),
            )

        fig.update_xaxes(
            range=[DATES[0], DATES[-1]],
            dtick="M24", tickformat="%Y",
            gridcolor="#E8EDF3",
        )
        fig.update_xaxes(title_text="Date", row=9, col=1)

        fig.add_annotation(
            x=-0.075, xref="paper",
            y=0.41, yref="paper",
            text="Radiance (nW·cm⁻²·sr⁻¹)",
            textangle=-90, showarrow=False,
            font=dict(size=12, color="#233B60"),
        )

        cap_label = (
            "raw radiance" if CAP_PERCENTILE is None
            else f"P{CAP_PERCENTILE} display cap"
        )
        fig.update_layout(
            template="plotly_white",
            width=1250, height=1050,
            title=dict(
                text=f"Daily DNB-BRDF by GHSL mask · {threshold_label}"
                     f" · 2013–2024 · {cap_label}",
                x=0.04,
            ),
            showlegend=False,
            font=dict(color="#233B60", size=11),
            margin=dict(l=110, r=220, t=90, b=65),
            paper_bgcolor="white",
            plot_bgcolor="white",
        )

        if display_figure:
            fig.show()

        tag = "none" if sc_threshold is None else str(sc_threshold)
        png = OUT_DIR / f"ghsl_daily_sc_{tag}_raw.png"
        export_figure(fig, Path(str(png)).with_suffix(""), scale=2)

        return coverage


    all_coverage = []
    for threshold in SC_THRESHOLDS:
        all_coverage.extend(plot_daily_masks(threshold, display_figure=(threshold == 50)))

    display(
        pd.DataFrame(all_coverage)
        .pivot(index="mask", columns="SC threshold", values="N%")
        .round(1)
    )

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    # Load the complete 2012–2024 RQ1 mask exports; NGCP scoring starts in 2013.
    SOURCE_DATES = pd.date_range('2012-01-19', '2024-12-31', freq='D', name='date')
    VALIDATION_DATES = pd.date_range('2013-01-01', '2024-12-31', freq='D', name='date')
    W_VALUES = np.arange(1, 31)
    DISPLAY_TAUS = np.arange(0., 101., 1.)  # only the display grid; selection uses exact SC cutoffs
    P95_ANALYSIS_CAP = 95                  # Notebook 03's 0th–95th percentile winsorisation
    SENS_DIR = TABLE_DIAGNOSTIC
    SENS_DIR.mkdir(parents=True, exist_ok=True)

    def full_mask_export(suffix):
        path = MASK_DIR / f'EasternVisayas_DNBBRDF_{suffix}_stats.csv'
        if not path.is_file():
            raise FileNotFoundError(path.relative_to(PROJECT_ROOT))
        d = pd.read_csv(path)
        if 'date' not in d.columns:
            if 'system:index' not in d.columns:
                raise ValueError(f'{path.name}: date and system:index are both absent')
            d['date'] = (d['system:index'].astype(str)
                         .str.extract(r'(\d{4}_\d{2}_\d{2})')[0]
                         .str.replace('_', '-', regex=False))
        d['date'] = pd.to_datetime(d['date'], errors='raise').dt.normalize()
        if d['date'].duplicated().any():
            raise ValueError(f'{path.name}: duplicate dates')
        d = d.set_index('date').sort_index()
        required = {'Valid_px', 'Mean_by_sum'}
        if not required.issubset(d.columns):
            raise ValueError(f'{path.name}: missing {sorted(required - set(d.columns))}')
        valid_px = pd.to_numeric(d['Valid_px'], errors='coerce')
        raw = pd.to_numeric(d['Mean_by_sum'], errors='coerce').where(valid_px > 0)
        denominator = valid_px.loc[raw.notna()].max()
        if not np.isfinite(denominator) or denominator <= 0:
            raise ValueError(f'{path.name}: no direct observations')
        sc = 100 * valid_px / denominator
        low, high = raw.quantile(0), raw.quantile(P95_ANALYSIS_CAP / 100)
        ntl = raw.clip(lower=low, upper=high)
        return (pd.DataFrame({'raw': raw, 'ntl': ntl, 'SC': sc, 'Valid_px': valid_px})
                .reindex(SOURCE_DATES)), {'source': path.relative_to(PROJECT_ROOT).as_posix(), 'SC_denominator': denominator,
                                          'P95_cap': high, 'direct_dates': int(raw.notna().sum())}

    SENS_MASK_DATA = {}
    source_rows = []
    for code, suffix, label, color in reversed(GROUPS):
        SENS_MASK_DATA[code], audit = full_mask_export(suffix)
        source_rows.append({'mask': code, 'label': label, **audit})
    SOURCE_AUDIT = pd.DataFrame(source_rows)
    display(SOURCE_AUDIT)
    SOURCE_AUDIT.to_csv(SENS_DIR / 'source_audit.csv', index=False)

    SENS_LOAD = LOAD.reindex(SOURCE_DATES).astype(float)
    VALIDATION_ENDPOINT = ((SOURCE_DATES >= VALIDATION_DATES[0])
                           & (SOURCE_DATES <= VALIDATION_DATES[-1])
                           & SENS_LOAD.notna().to_numpy())
    N_POSSIBLE = int(VALIDATION_ENDPOINT.sum())
    if N_POSSIBLE == 0:
        raise ValueError('No observed NGCP 1 AM dates in 2013–2024')
    print(f'NGCP comparison dates, 2013–2024: {N_POSSIBLE:,}')

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    def paired_alignment(x, y):
        paired = pd.DataFrame({'ntl': x, 'load': y}).reindex(SOURCE_DATES)
        paired = paired.loc[VALIDATION_ENDPOINT].dropna()
        n = len(paired)
        r = np.nan
        if n >= 3 and paired.ntl.std(ddof=0) > 0 and paired.load.std(ddof=0) > 0:
            r = float(paired.ntl.corr(paired.load))
        return {'n_pairs': n, 'N_pct': 100 * n / N_POSSIBLE, 'r': r,
                'RMSE_z': float(np.sqrt(max(0, 2 * (1 - r)))) if np.isfinite(r) else np.nan}

    AUDIT_MASK, AUDIT_TAU, AUDIT_W = 'G3', 10., 30
    aud = SENS_MASK_DATA[AUDIT_MASK]
    accepted = aud.ntl.notna() & aud.SC.ge(AUDIT_TAU)
    legacy_ntl = aud.loc[accepted, 'ntl'].rolling(AUDIT_W, min_periods=1, center=True).mean()
    calendar_ntl = aud.ntl.where(accepted).rolling(AUDIT_W, min_periods=1, center=True).mean()
    calendar_load = SENS_LOAD.rolling(AUDIT_W, min_periods=1, center=True).mean()

    METHOD_AUDIT = pd.DataFrame([
        {'method': 'Original: 30 retained NTL observations',
         **paired_alignment(legacy_ntl, calendar_load)},
        {'method': 'Corrected: 30 calendar days for both',
         **paired_alignment(calendar_ntl, calendar_load)},
    ])
    display(METHOD_AUDIT.round(3))
    METHOD_AUDIT.to_csv(SENS_DIR / 'legacy_vs_calendar_30_day_audit.csv', index=False)

    zoom = pd.date_range('2013-05-01', '2014-05-31', freq='D')
    f = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=.09,
                      row_heights=[.17, .83], specs=[[{}], [{'secondary_y': True}]],
                      subplot_titles=['SC (%)', 'Centered 30-day means'])
    f.add_trace(go.Heatmap(x=zoom, y=[0], z=[aud.SC.reindex(zoom).fillna(0).to_numpy()],
                           zmin=0, zmax=100, colorscale='Greens',
                           colorbar=dict(title='SC %', len=.33, y=.93, x=1.01),
                           hovertemplate='%{x|%d %b %Y}<br>SC %{z:.1f}%<extra></extra>'), row=1, col=1)
    for series, name, color, dash in [
        (legacy_ntl, 'Original: 30 retained NTL observations', '#D65A4A', 'dot'),
        (calendar_ntl, 'NTL: 30 calendar days', '#2952CC', 'solid'),
    ]:
        f.add_trace(go.Scatter(x=zoom, y=series.reindex(zoom), mode='lines', name=name,
                               line=dict(color=color, width=2, dash=dash), connectgaps=False), row=2, col=1)
    f.add_trace(go.Scatter(x=zoom, y=calendar_load.reindex(zoom), mode='lines',
                           name='NGCP: 30 calendar days',
                           line=dict(color='#202020', width=2)), row=2, col=1,
                secondary_y=True)
    f.add_vline(x=pd.Timestamp('2013-11-08'), line_dash='dash', line_color='#233B60')
    f.update_yaxes(title_text='Radiance', row=2, col=1, secondary_y=False)
    f.update_yaxes(title_text='NGCP load (MW)', row=2, col=1, secondary_y=True,
                   showgrid=False)
    f.update_yaxes(showticklabels=False, row=1, col=1)
    f.update_layout(template='plotly_white', width=1200, height=500,
                    title='Why the old “30 days” was not a common calendar window',
                    legend=dict(orientation='h', y=-.18),
                    margin=dict(l=75, r=100, t=80, b=95), font=dict(color='#233B60'))
    f.show()
    export_figure(f, Path(FIG_DIAGNOSTIC / 'original_vs_calendar_30_day.html').with_suffix(""), static=True)

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    PROFILE_GROUPS = ("G1", "G8")
    PROFILE_WINDOWS = (1, 4, 7, 14, 21, 30)
    PROFILE_LEFT = pd.Timestamp("2013-05-01")
    PROFILE_RIGHT = pd.Timestamp("2014-05-31")
    HAIYAN_DATE = pd.Timestamp("2013-11-08")

    GROUP_STYLE = {
        code: (label, color)
        for code, suffix, label, color in GROUPS
    }
    profile_dates = pd.date_range(
        PROFILE_LEFT, PROFILE_RIGHT, freq="D"
    )
    daily_load = SENS_LOAD.reindex(SOURCE_DATES).astype(float)
    load_limit = 1.08 * float(
        daily_load.reindex(profile_dates).max()
    )

    PROFILE_FIGURES = {}
    profile_rows = []

    for code in PROFILE_GROUPS:
        label, color = GROUP_STYLE[code]

        # Every available direct date; no SC threshold.
        direct_ntl = (
            SENS_MASK_DATA[code]["ntl"]
            .reindex(SOURCE_DATES)
            .astype(float)
        )
        ntl_limit = 1.08 * float(
            direct_ntl.reindex(profile_dates).max()
        )

        fig = make_subplots(
            rows=2,
            cols=3,
            specs=[
                [{"secondary_y": True} for _ in range(3)]
                for _ in range(2)
            ],
            shared_xaxes=True,
            horizontal_spacing=.085,
            vertical_spacing=.16,
            subplot_titles=[
                f"W={width} d" for width in PROFILE_WINDOWS
            ],
        )
        for heading in fig.layout.annotations:
            heading.font = dict(
                size=16, color="#233B60"
            )

        panel_legends = {}

        for i, width in enumerate(PROFILE_WINDOWS):
            row, col = i // 3 + 1, i % 3 + 1
            legend_id = (
                "legend" if i == 0 else f"legend{i + 1}"
            )

            ntl_mean = direct_ntl.rolling(
                width, min_periods=1, center=True
            ).mean()
            load_mean = daily_load.rolling(
                width, min_periods=1, center=True
            ).mean()

            metrics = paired_alignment(
                ntl_mean, load_mean
            )
            profile_rows.append({
                "mask": code,
                "window_days": width,
                **metrics,
            })

            r_text = (
                f"{metrics['r']:.2f}"
                if np.isfinite(metrics["r"]) else "NA"
            )
            ntl_legend = (
                f"{code} · r={r_text} · "
                f"N%={metrics['N_pct']:.1f}"
            )

            # The direct observations remain visible behind the mean.
            fig.add_trace(
                go.Scatter(
                    x=profile_dates,
                    y=direct_ntl.reindex(profile_dates),
                    mode="markers",
                    name=ntl_legend,
                    legend=legend_id,
                    showlegend=(width == 1),
                    marker=dict(
                        color=color,
                        size=4,
                        opacity=.55 if width == 1 else .22,
                    ),
                ),
                row=row, col=col, secondary_y=False,
            )

            fig.add_trace(
                go.Scatter(
                    x=profile_dates,
                    y=ntl_mean.reindex(profile_dates),
                    mode="lines",
                    name=ntl_legend,
                    legend=legend_id,
                    showlegend=(width != 1),
                    connectgaps=False,
                    line=dict(color=color, width=2.2),
                ),
                row=row, col=col, secondary_y=False,
            )

            fig.add_trace(
                go.Scatter(
                    x=profile_dates,
                    y=load_mean.reindex(profile_dates),
                    mode="lines",
                    name="NGCP 1 AM",
                    legend=legend_id,
                    showlegend=True,
                    connectgaps=False,
                    line=dict(
                        color="#202020", width=1.8
                    ),
                ),
                row=row, col=col, secondary_y=True,
            )

            fig.add_vline(
                x=HAIYAN_DATE,
                row=row, col=col,
                line_color="#233B60",
                line_dash="dash",
                line_width=1.2,
            )

            fig.update_xaxes(
                range=[PROFILE_LEFT, PROFILE_RIGHT],
                dtick="M2",
                tickformat="%b %Y",
                tickangle=-30,
                showticklabels=True,
                gridcolor="#E8EDF3",
                row=row, col=col,
            )
            fig.update_yaxes(
                range=[0, ntl_limit],
                showticklabels=(col == 1),
                title_text=(
                    "Radiance (nW·cm⁻²·sr⁻¹)"
                    if col == 1 else None
                ),
                gridcolor="#E8EDF3",
                row=row, col=col,
                secondary_y=False,
            )
            fig.update_yaxes(
                range=[0, load_limit],
                showticklabels=(col == 3),
                title_text=(
                    "NGCP load (MW)"
                    if col == 3 else None
                ),
                showgrid=False,
                row=row, col=col,
                secondary_y=True,
            )

            # Position this panel's own two-item legend inside it.
            xaxis_key = (
                "xaxis" if i == 0 else f"xaxis{i + 1}"
            )
            primary_y_key = (
                "yaxis" if i == 0
                else f"yaxis{2 * i + 1}"
            )
            x_domain = fig.layout[xaxis_key].domain
            y_domain = fig.layout[primary_y_key].domain

            panel_legends[legend_id] = dict(
                x=x_domain[1] - .01,
                y=y_domain[1] - .01,
                xanchor="right",
                yanchor="top",
                bgcolor="rgba(255,255,255,.85)",
                font=dict(size=10),
                borderwidth=0,
            )

        fig.update_layout(
            template="plotly_white",
            width=1550,
            height=820,
            title=dict(
                text=f"{code} · {label} | no SC threshold",
                x=.02,
                font=dict(size=17),
            ),
            margin=dict(
                l=95, r=95, t=85, b=95
            ),
            font=dict(
                family="Arial",
                color="#233B60",
                size=12,
            ),
            paper_bgcolor="white",
            plot_bgcolor="white",
            **panel_legends,
        )

        fig.add_annotation(
            x=.5, y=-.12,
            xref="paper", yref="paper",
            text=(
                "Lines: Haiyan zoom · "
                "Legend r and N%: full 2013–2024 comparison"
            ),
            showarrow=False,
            font=dict(size=11, color="#233B60"),
        )

        fig.show()
        export_figure(fig, Path(FIG_DIAGNOSTIC / f"{code.lower()}_six_window_profiles.html").with_suffix(""), static=True)
        PROFILE_FIGURES[code] = fig

    PROFILE_METRICS = pd.DataFrame(profile_rows)

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    from time import perf_counter

    SMOOTHED_LOAD = {
        w: SENS_LOAD.rolling(w, min_periods=1, center=True).mean().to_numpy(dtype=float)
        for w in W_VALUES
    }

    def exact_threshold_sweep(code, width):
        d = SENS_MASK_DATA[code]
        x = d.ntl.to_numpy(dtype=float)
        sc = d.SC.to_numpy(dtype=float)
        observed = np.isfinite(x) & np.isfinite(sc)
        values = np.where(observed, x, 0.)
        y = SMOOTHED_LOAD[width]
        endpoint_ok = VALIDATION_ENDPOINT & np.isfinite(y)
        size = len(x)

        # Start with every observed direct day admitted (τ=0).
        count = (pd.Series(observed.astype(int), index=SOURCE_DATES)
                 .rolling(width, min_periods=1, center=True).sum()
                 .to_numpy(dtype=int))
        total = (pd.Series(values, index=SOURCE_DATES)
                 .rolling(width, min_periods=1, center=True).sum()
                 .to_numpy(dtype=float))
        mean = np.divide(total, count, out=np.zeros(size), where=count > 0)
        pair = endpoint_ok & (count > 0)
        xx, yy = mean[pair], y[pair]
        n = int(pair.sum())
        sx, sy = float(xx.sum()), float(yy.sum())
        sxx, syy, sxy = float((xx * xx).sum()), float((yy * yy).sum()), float((xx * yy).sum())
        support_sum = float(count[pair].sum())
        direct_n = int((observed & VALIDATION_ENDPOINT).sum())

        days = np.flatnonzero(observed)
        days = days[np.argsort(sc[days], kind='stable')]
        taus = np.unique(np.r_[0., 100., sc[days]])
        position = 0
        rows = []
        label = next(label for c, suffix, label, color in GROUPS if c == code)

        for tau in taus:
            # Move from one exact SC cutoff to the next by removing only
            # observations that no longer qualify. Each affects at most W endpoints.
            while position < len(days) and sc[days[position]] < tau:
                i = int(days[position])
                position += 1
                lo = max(0, i - (width - 1) // 2)
                hi = min(size, i + width // 2 + 1)
                old_count = count[lo:hi].copy()
                old_total = total[lo:hi].copy()
                segment_y = y[lo:hi]
                allowed = endpoint_ok[lo:hi]
                old_mean = np.divide(old_total, old_count,
                                     out=np.zeros(len(old_count)), where=old_count > 0)
                old_pair = (old_count > 0) & allowed
                old_x, old_y = old_mean[old_pair], segment_y[old_pair]
                n -= int(old_pair.sum())
                sx -= float(old_x.sum()); sy -= float(old_y.sum())
                sxx -= float((old_x * old_x).sum()); syy -= float((old_y * old_y).sum())
                sxy -= float((old_x * old_y).sum())
                support_sum -= float(old_count[old_pair].sum())

                new_count = old_count - 1
                new_total = old_total - x[i]
                new_total[new_count == 0] = 0.
                count[lo:hi] = new_count
                total[lo:hi] = new_total
                new_mean = np.divide(new_total, new_count,
                                     out=np.zeros(len(new_count)), where=new_count > 0)
                new_pair = (new_count > 0) & allowed
                new_x, new_y = new_mean[new_pair], segment_y[new_pair]
                n += int(new_pair.sum())
                sx += float(new_x.sum()); sy += float(new_y.sum())
                sxx += float((new_x * new_x).sum()); syy += float((new_y * new_y).sum())
                sxy += float((new_x * new_y).sum())
                support_sum += float(new_count[new_pair].sum())
                if VALIDATION_ENDPOINT[i]:
                    direct_n -= 1

            r = np.nan
            if n >= 3:
                vx = sxx - sx * sx / n
                vy = syy - sy * sy / n
                tol_x = np.finfo(float).eps * max(1., abs(sxx))
                tol_y = np.finfo(float).eps * max(1., abs(syy))
                if vx > tol_x and vy > tol_y:
                    r = float(np.clip((sxy - sx * sy / n) / np.sqrt(vx * vy), -1, 1))
            rmse_z = float(np.sqrt(max(0, 2 * (1 - r)))) if np.isfinite(r) else np.nan
            n_fraction = n / N_POSSIBLE
            score = n_fraction * max(r, 0)**2 if np.isfinite(r) else 0.
            rows.append((code, label, int(width), float(tau), n, N_POSSIBLE,
                         100 * n_fraction, direct_n, 100 * direct_n / N_POSSIBLE,
                         support_sum / n if n else np.nan,
                         100 * support_sum / (n * width) if n else np.nan,
                         r, rmse_z, score))
        return rows

    start_sweep = perf_counter()
    sweep_rows = []
    for code, suffix, label, color in reversed(GROUPS):
        for width in W_VALUES:
            sweep_rows.extend(exact_threshold_sweep(code, int(width)))

    SENSITIVITY = pd.DataFrame(sweep_rows, columns=[
        'mask', 'label', 'window_days', 'tau', 'n_pairs', 'n_possible',
        'N_pct', 'direct_n', 'direct_pct', 'mean_direct_days',
        'mean_window_support_pct', 'r', 'RMSE_z', 'score',
    ])
    SENSITIVITY.to_csv(SENS_DIR / 'complete_mask_sc_window_surface.csv.gz',
                       compression='gzip', index=False)
    print(f'{len(SENSITIVITY):,} exact configurations evaluated in '
          f'{(perf_counter() - start_sweep):.1f} seconds.')
    print('Full exact table:', (SENS_DIR / 'complete_mask_sc_window_surface.csv.gz').relative_to(PROJECT_ROOT))
    display(SENSITIVITY.groupby('mask').agg(
        configurations=('tau', 'size'),
        distinct_tau=('tau', 'nunique'),
        windows=('window_days', 'nunique'),
    ))

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    DAILY = SENSITIVITY[
        (SENSITIVITY.window_days == 1) & SENSITIVITY.r.notna()
    ].copy()
    if DAILY.empty:
        raise ValueError("No native-daily configuration has a defined Pearson r")

    DAILY = DAILY.sort_values(
        ["score", "N_pct", "tau", "mask"],
        ascending=[False, False, True, True],
    )
    WINNER = DAILY.iloc[0]
    DAILY_TOP = DAILY.head(12).copy()
    DAILY_TOP.to_csv(SENS_DIR / "top_native_daily_settings.csv", index=False)

    print("Top native-daily mask × SC settings:")
    display(
        DAILY_TOP[[
            "mask", "label", "tau", "r", "n_pairs", "N_pct",
            "direct_pct", "score",
        ]].round(3)
    )

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    SUPPORT_RULES = [(None, 'At least 1 day'), (.25, 'At least 25%'),
                     (.50, 'At least 50%'), (.75, 'At least 75%'),
                     (1.0, 'All days')]
    selected_mask_data = SENS_MASK_DATA[WINNER['mask']]
    selected_daily = selected_mask_data.ntl.where(selected_mask_data.SC >= WINNER.tau)
    support_rows = []
    for width in W_VALUES:
        count = selected_daily.notna().astype(int).rolling(
            int(width), min_periods=1, center=True).sum()
        ntl_mean = selected_daily.rolling(int(width), min_periods=1, center=True).mean()
        load_mean = SENS_LOAD.rolling(int(width), min_periods=1, center=True).mean()
        for fraction, rule_label in SUPPORT_RULES:
            needed = 1 if fraction is None else int(np.ceil(fraction * width))
            x = ntl_mean.where(count >= needed)
            stats = paired_alignment(x, load_mean)
            support_rows.append({'rule': rule_label, 'min_direct_days': needed,
                                 'window_days': int(width), **stats})
    SUPPORT_CHECK = pd.DataFrame(support_rows)
    SUPPORT_CHECK.to_csv(SENS_DIR / 'selected_setting_support_rule_sensitivity.csv', index=False)
    display(SUPPORT_CHECK[SUPPORT_CHECK.window_days.isin([4, 7, 14, 30])]
            [['rule', 'window_days', 'min_direct_days', 'r', 'RMSE_z', 'N_pct']].round(3))

    rule_colors = ['#233B60', '#4E79A7', '#E6A23C', '#D65A4A', '#375623']
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=.12,
                        subplot_titles=['Pearson r', 'N% paired smoothed estimates'])
    for (fraction, label), color in zip(SUPPORT_RULES, rule_colors):
        q = SUPPORT_CHECK[SUPPORT_CHECK.rule == label].sort_values('window_days')
        for row, metric in [(1, 'r'), (2, 'N_pct')]:
            fig.add_trace(go.Scatter(x=q.window_days, y=q[metric], mode='lines',
                                     name=label, showlegend=(row == 1),
                                     line=dict(color=color, width=2.5), connectgaps=False),
                          row=row, col=1)
    fig.update_xaxes(title_text='Common calendar window (days)',
                     tickvals=[1, 4, 7, 14, 21, 30], row=2, col=1)
    fig.update_yaxes(title_text='Pearson r', row=1, col=1)
    fig.update_yaxes(title_text='N% paired estimates', range=[0, 100], row=2, col=1)
    fig.update_layout(template='plotly_white', width=1150, height=720,
                      title=f"Within-window direct support · {WINNER['mask']} · SC τ={WINNER.tau:.2f}%",
                      legend=dict(x=1.01, y=1), margin=dict(l=75, r=190, t=80, b=70),
                      font=dict(color='#233B60', size=11))
    fig.show()
    export_figure(fig, Path(FIG_DIAGNOSTIC / 'selected_setting_support_rule_sensitivity.html').with_suffix(""), static=True)

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    chosen = SENS_MASK_DATA[WINNER['mask']]
    chosen_direct = chosen.ntl.where(chosen.SC >= WINNER.tau)
    year_rows = []
    for width in W_VALUES:
        xx = chosen_direct.rolling(int(width), min_periods=1, center=True).mean()
        yy = SENS_LOAD.rolling(int(width), min_periods=1, center=True).mean()
        for year in range(2013, 2025):
            original_load = SENS_LOAD.loc[str(year)]
            possible = int(original_load.notna().sum())
            frame = pd.DataFrame({'x': xx.loc[str(year)], 'y': yy.loc[str(year)],
                                  'load_original': original_load}).dropna()
            n = len(frame)
            r = (float(frame.x.corr(frame.y)) if n >= 3
                 and frame.x.std(ddof=0) > 0 and frame.y.std(ddof=0) > 0 else np.nan)
            year_rows.append({'year': year, 'window_days': int(width), 'n_pairs': n,
                              'N_pct': 100 * n / possible if possible else np.nan,
                              'r': r})
    YEAR_CHECK = pd.DataFrame(year_rows)
    YEAR_CHECK.to_csv(SENS_DIR / 'selected_mask_tau_year_by_window.csv', index=False)

    fig = make_subplots(rows=1, cols=2, horizontal_spacing=.12,
                        subplot_titles=['Pearson r within year', 'N% paired within year'])
    for col, (metric, axis, palette, low, high) in enumerate([
        ('r', 'coloraxis', [[0, '#B2182B'], [.5, '#F7F7F7'], [1, '#2166AC']], -1, 1),
        ('N_pct', 'coloraxis2', 'Blues', 0, 100),
    ], 1):
        z = YEAR_CHECK.pivot(index='year', columns='window_days', values=metric)
        fig.add_trace(go.Heatmap(x=z.columns, y=z.index, z=z.values,
                                 coloraxis=axis, hoverongaps=False,
                                 hovertemplate=(f'Year %{{y}} · W %{{x}} d<br>'
                                                f'{metric} %{{z:.3f}}<extra></extra>')),
                      row=1, col=col)
    fig.update_xaxes(title_text='Common window (days)', tickvals=[1, 4, 7, 14, 21, 30])
    fig.update_yaxes(title_text='Year', dtick=1)
    fig.update_layout(template='plotly_white', width=1350, height=520,
                      title=f"Year-by-year check · {WINNER['mask']} · SC τ={WINNER.tau:.2f}%",
                      coloraxis=dict(colorscale=[[0, '#B2182B'], [.5, '#F7F7F7'],
                                                [1, '#2166AC']], cmin=-1, cmax=1,
                                     colorbar=dict(title='r', x=.45, len=.8)),
                      coloraxis2=dict(colorscale='Blues', cmin=0, cmax=100,
                                      colorbar=dict(title='N%', x=1.02, len=.8)),
                      margin=dict(l=70, r=130, t=85, b=65),
                      font=dict(color='#233B60', size=11))
    fig.show()
    export_figure(fig, Path(FIG_DIAGNOSTIC / 'selected_setting_year_stability.html').with_suffix(""), static=True)

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    WINDOW_TAUS = (0, 25, 50, 75)
    WINDOW_DAYS = range(1, 31)
    WINDOW_CHECKPOINTS = (1, 4, 7, 14, 21, 30)
    WINDOW_TABLE_DIR = TABLE_DIAGNOSTIC
    WINDOW_TABLE_DIR.mkdir(parents=True, exist_ok=True)

    n_possible = int(LOAD.notna().sum())
    if n_possible == 0:
        raise ValueError("No 2013–2024 NGCP 1 AM dates were loaded")

    window_rows = []
    for code, suffix, label, color in reversed(GROUPS):
        d = MASK_DATA[code]
        raw = d["raw"]
        if CAP_PERCENTILE is None:
            radiance = raw
        else:
            radiance = raw.clip(upper=raw.quantile(CAP_PERCENTILE / 100))

        for tau in WINDOW_TAUS:
            admitted = raw.notna() & (d["SC"] >= tau)
            direct = radiance.where(admitted)
            for width in WINDOW_DAYS:
                # min_periods=width: every day in this trailing calendar window
                # must have a direct SC-qualified NTL value and an NGCP value.
                ntl_window = direct.rolling(width, min_periods=width).mean()
                load_window = LOAD.rolling(width, min_periods=width).mean()
                paired = pd.concat(
                    [ntl_window.rename("ntl"), load_window.rename("load")], axis=1
                ).dropna()
                n = len(paired)
                r = np.nan
                if n < 3:
                    r_status = "fewer than 3 paired dates"
                elif (paired["ntl"].std(ddof=0) == 0
                      or paired["load"].std(ddof=0) == 0):
                    r_status = "constant paired series"
                else:
                    r = float(paired["ntl"].corr(paired["load"]))
                    r_status = "defined"
                window_rows.append({
                    "mask": code, "label": label, "tau": tau,
                    "window_days": width, "r": r, "r_status": r_status,
                    "n_pairs": n, "n_possible": n_possible,
                    "n_pct": 100 * n / n_possible,
                })

    WINDOW_METRICS = pd.DataFrame(window_rows)
    for _, group in WINDOW_METRICS.groupby(["mask", "tau"]):
        counts = group.sort_values("window_days")["n_pairs"]
        assert counts.diff().dropna().le(0).all(), "Complete-window N must not rise with width"

    WINDOW_METRICS.to_csv(
        WINDOW_TABLE_DIR / "ghsl_complete_window_tau0_25_50_75_2013_2024.csv", index=False
    )
    WINDOW_CHECKPOINT_TABLE = WINDOW_METRICS[
        WINDOW_METRICS.window_days.isin(WINDOW_CHECKPOINTS)
    ].copy()
    WINDOW_CHECKPOINT_TABLE.to_csv(
        WINDOW_TABLE_DIR / "ghsl_window_checkpoints_tau0_25_50_75.csv", index=False
    )
    WINDOW_BREAKS = (WINDOW_METRICS[WINDOW_METRICS.r.isna()]
                     .sort_values("window_days")
                     .groupby(["tau", "mask"], as_index=False).first()
                     [["tau", "mask", "window_days", "n_pairs", "r_status"]])
    WINDOW_BREAKS.to_csv(
        WINDOW_TABLE_DIR / "ghsl_first_undefined_r_by_mask_tau.csv", index=False
    )
    print(f"Saved {len(WINDOW_METRICS)} mask × SC × window results; NGCP denominator = {n_possible} days")
    display(WINDOW_CHECKPOINT_TABLE[
        WINDOW_CHECKPOINT_TABLE["mask"].isin(["G1", "G5", "G8"])
    ][["mask", "tau", "window_days", "r", "r_status", "n_pairs", "n_pct"]].round(3))
    display(WINDOW_BREAKS)

In [ ]:
if RUN_ADDITIONAL_DIAGNOSTICS:
    WINDOW_COLOR = {code: color for code, suffix, label, color in GROUPS}
    LABEL_WINDOWS = (1, 4, 30)

    FIG_WIDTH, FIG_HEIGHT = 1500, 470
    FIG_MARGIN = dict(l=75, r=230, t=100, b=95)


    def plot_window_metric(metric, filename):
        if metric not in ("r", "n_pct"):
            raise ValueError(metric)

        is_r = metric == "r"
        fig = make_subplots(
            rows=1,
            cols=4,
            shared_yaxes=True,
            horizontal_spacing=0.055,
        )

        for col, tau in enumerate(WINDOW_TAUS, start=1):
            panel = WINDOW_METRICS[WINDOW_METRICS.tau == tau]
            axis_x = "x" if col == 1 else f"x{col}"
            axis_y = "y" if col == 1 else f"y{col}"
            marker_points = []

            fig.add_annotation(
                x=15, y=1.17,
                xref=axis_x, yref="paper",
                text=f"SC τ={tau}%",
                showarrow=False,
                font=dict(size=19, color="#233B60"),
            )

            for code in ("G1", "G8"):
                color = WINDOW_COLOR[code]

                for window in LABEL_WINDOWS:
                    q = panel[
                        (panel["mask"] == code)
                        & (panel.window_days == window)
                    ].iloc[0]
                    value = float(q[metric])

                    if (
                        not np.isfinite(value)
                        or (is_r and value < 0)
                        or (not is_r and value <= 0)
                    ):
                        continue

                    # Plotly annotations on a log axis use log10 positions.
                    point_y = value if is_r else np.log10(value)

                    # Fixed positions requested: no automatic flipping.
                    above = (
                        code == "G1"
                        if is_r else code == "G8"
                    )

                    fig.add_annotation(
                        x=window,
                        y=point_y,
                        xref=axis_x,
                        yref=axis_y,
                        text=(
                            f"{value:.2f}"
                            if is_r else f"{value:.1f}%"
                        ),
                        showarrow=True,
                        arrowhead=0,
                        arrowwidth=0.75,
                        arrowcolor="#A9B9C9",
                        ax=0,
                        ay=-22 if above else 22,
                        xanchor=(
                            "right" if window == 30 else "center"
                        ),
                        yanchor="bottom" if above else "top",
                        font=dict(size=14, color=color),
                        bgcolor="white",
                        bordercolor=color,
                        borderwidth=0.8,
                        borderpad=2,
                    )

                    marker_points.append((window, value, color))

            for code, suffix, label, color in reversed(GROUPS):
                q = panel[
                    panel["mask"] == code
                ].sort_values("window_days")

                custom = np.column_stack([
                    q.n_pairs.to_numpy(),
                    q.n_pct.to_numpy(),
                ])
                hover = (
                    "Window %{x} d<br>"
                    "Value %{y:.3f}<br>"
                    "Paired dates %{customdata[0]:.0f}<br>"
                    "N %{customdata[1]:.2f}%"
                    "<extra>%{fullData.name}</extra>"
                )

                if is_r:
                    # Negative r is hidden only in this positive-alignment view.
                    # Undefined r stays NaN, creating a genuine break.
                    shown_r = q.r.where(q.r >= 0)

                    fig.add_trace(
                        go.Scatter(
                            x=q.window_days,
                            y=shown_r,
                            mode="lines",
                            name=f"{code} {label}",
                            legendgroup=code,
                            showlegend=(col == 1),
                            line=dict(color=color, width=2.2),
                            connectgaps=False,
                            customdata=custom,
                            hovertemplate=hover,
                        ),
                        row=1, col=col,
                    )
                else:
                    # Zero cannot be drawn on a log axis.
                    shown_n = q.n_pct.where(q.n_pct > 0)

                    fig.add_trace(
                        go.Scatter(
                            x=q.window_days,
                            y=shown_n,
                            mode="lines",
                            name=f"{code} {label}",
                            legendgroup=code,
                            showlegend=(col == 1),
                            line=dict(color=color, width=2.2),
                            connectgaps=False,
                            customdata=custom,
                            hovertemplate=hover,
                        ),
                        row=1, col=col,
                    )

            # Draw checkpoint dots after the curves so they remain visible.
            for window, value, color in marker_points:
                fig.add_trace(
                    go.Scatter(
                        x=[window],
                        y=[value],
                        mode="markers",
                        marker=dict(
                            color=color,
                            size=7,
                            line=dict(color="white", width=1),
                        ),
                        showlegend=False,
                        hoverinfo="skip",
                    ),
                    row=1, col=col,
                )

            fig.update_xaxes(
                range=[1, 30],
                tickvals=[1, 4, 7, 14, 21, 30],
                tickfont=dict(size=15),
                gridcolor="#E8EDF3",
                showline=True,
                linecolor="#233B60",
                linewidth=1.4,
                ticks="outside",
                ticklen=4,
                tickwidth=1.2,
                row=1, col=col,
            )

            if is_r:
                fig.update_yaxes(
                    type="linear",
                    range=[-0.1, 1.1],
                    tickvals=[0, 0.25, 0.5, 0.75, 1],
                    showticklabels=(col == 1),
                    tickfont=dict(size=15),
                    gridcolor="#E8EDF3",
                    zeroline=False,
                    showline=True,
                    linecolor="#233B60",
                    linewidth=1.4,
                    ticks="outside",
                    ticklen=4,
                    tickwidth=1.2,
                    row=1, col=col,
                )
            else:
                fig.update_yaxes(
                    type="log",
                    range=[-2, 2],
                    tickvals=[0.01, 0.1, 1, 10, 100],
                    ticktext=["0.01", "0.1", "1", "10", "100"],
                    showticklabels=(col == 1),
                    tickfont=dict(size=15),
                    gridcolor="#E8EDF3",
                    showline=True,
                    linecolor="#233B60",
                    linewidth=1.4,
                    ticks="outside",
                    ticklen=4,
                    tickwidth=1.2,
                    row=1, col=col,
                )

        fig.update_yaxes(
            title_text=(
                "Positive Pearson r"
                if is_r else "N% retained coverage (log scale)"
            ),
            title_font=dict(size=18),
            row=1, col=1,
        )

        fig.add_annotation(
            x=0.5, y=-0.28,
            xref="paper", yref="paper",
            text="Calendar window (days)",
            showarrow=False,
            font=dict(size=18, color="#233B60"),
        )

        fig.update_layout(
            template="plotly_white",
            width=FIG_WIDTH,
            height=FIG_HEIGHT,
            legend=dict(
                x=1.015, y=1,
                xanchor="left", yanchor="top",
                font=dict(size=15),
            ),
            margin=FIG_MARGIN,
            font=dict(
                family="Arial",
                color="#233B60",
                size=17,
            ),
            paper_bgcolor="white",
            plot_bgcolor="white",
        )

        fig.show()
        
        export_figure(fig, Path(str(OUT_DIR / f"{filename}.png")).with_suffix(""), scale=2)

        return fig


    WINDOW_R_FIG = plot_window_metric(
        "r", "ghsl_r_window_tau0_25_50_75"
    )
    WINDOW_N_FIG = plot_window_metric(
        "n_pct", "ghsl_n_window_tau0_25_50_75"
    )

## 7. Outputs

All figures use outputs/figures/ and all tables use outputs/tables/. Historical and exploratory sensitivity exports use their diagnostics/ subfolders.

## Conclusion

Window-supported estimates can increase apparent coverage and change signal shape. Interpret those effects separately from direct observation availability. Score-form comparisons show whether configuration selection depends on the chosen emphasis on alignment.